# N2 - evaluate, calibrate the certificates, test once

Evaluate and certify.  GPU only for the sliding-window inference; everything else is CPU.

Order matters and is enforced by the cell order: (1) probabilities for val + test are
cached once; (2) post-processing is tuned on VALIDATION only and frozen; (3) the risk
thresholds are CALIBRATED on validation (missed-tumour <= alpha, wrong-prediction <= beta)
and the implied volume interval is derived -- pooled AND stratified by image contrast, with
the per-stratum guarantee re-checked on test at each stratum's own threshold; (4) test is
touched exactly once, after all of that is frozen.


In [16]:
!pip install monai


In [17]:
# %% [CELL 2] SETTINGS -------------------------------------------------------
import os

ARMS        = ["base", "sr50"]   # must match the finished runs in the attached dataset
BASELINE    = "base"             # the control every arm is compared against
PRIMARY     = "wt"               # region the certificate is about: wt | tc | et
ALPHAS      = [0.05, 0.10, 0.20] # tolerated share of TRUE tumour that may be missed
BETAS       = [0.10, 0.20, 0.30] # tolerated share of the PREDICTION that may be wrong
LAMBDA_GRID = None               # None -> np.linspace(0.05, 0.95, 19)
INFER_OVERLAP = 0.5              # sliding-window overlap for the final inference
POST_THR_GRID = [0.40, 0.50, 0.60]
POST_CC_GRID  = [0, 60, 150, 400]        # minimum component size in mm^3 (1 mm voxels)
POST_ET_GRID  = [0, 150, 400, 800]       # ET-relabel-to-necrosis threshold in mm^3
POST_REGION_THR_GRID = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]
                                 # per-region TC/ET thresholds, tuned on VALIDATION only,
                                 # one common set for every arm.  WT keeps the certified
                                 # lambda, so this cannot change the certificate.  Set to []
                                 # to fall back to the old single-threshold behaviour.
TTA_FLIPS     = [(), (0,), (1,), (2,)]
                                 # flip averaging at inference: identity + one flip per axis.
                                 # Each extra entry = one more sliding-window pass per case, no
                                 # training.  Set to [] or [()] to disable (and to reuse a
                                 # probability cache computed without it).
ENSEMBLE      = [("ens", ("base", "sr50"))]
                                 # derived probability maps = mean of the named arms.  Same
                                 # weights, no extra inference (the arms are already cached), and
                                 # the averaging happens on PROBABILITIES, before any threshold.
                                 # [] to disable.  These appear as extra keys in certificates.json
                                 # / summary.json and are labelled as delivery candidates.
ARM_PREFERENCE = ["ens", "base", "sr50"]
                                 # which probabilities the delivered artefacts (report page, N3)
                                 # should show -- first one present wins
POST_OBJ_W    = dict(dice=1.0, lesion=0.5, volume=0.5)
                                 # what post-processing is tuned FOR.  Mean voxel Dice alone
                                 # rewards big blobs, which is how a filter that deletes real
                                 # satellites still scores well.  The objective is
                                 #   dice*mean_region_dice + lesion*mean_lesionwise_dice
                                 #   - volume*mean(|pred-gt|/gt)
                                 # all three components are written to post_grid.csv.
POST_TUNE_ARM = "auto"           # "auto" -> the first ARM_PREFERENCE entry that exists, else
                                 # the control arm.  One common set is still used by EVERY arm.
CERTIFIED_FILTER = True          # min component size (voxels) per region comes from the
                                 # certified filter (collateral-damage bound on validation)
                                 # instead of a Dice grid search.  False -> tune it like before.
FORCE_POST    = None             # dict(min_vox={...}, min_et_voxels=..) to skip tuning
FORCE_ALPHA   = None             # e.g. 0.10 to pin the certificate instead of using the grid
FORCE_BETA    = None             # e.g. 0.20
RUN_TEST      = True             # test is touched ONCE, after everything is frozen
FILTER_ALPHAS = [0.01, 0.05, 0.10]       # certified component-filter levels
FILTER_GRID   = [1, 30, 80, 150, 300, 600]
PATCH       = (96, 96, 96)
WORK        = "/kaggle/working"
INPUT_ROOT  = os.environ.get("BRATS_INPUT_ROOT") or "/kaggle/input"
                                 # where Kaggle mounts attached datasets / notebook outputs
CACHE_BASE  = None               # None = auto-discover the attached cache shard(s)
RUNS_INPUT  = None               # None = auto-discover the attached runs dataset
ALLOW_NONFINITE_CASES = False   # a probability map that is nan/inf compares as an EMPTY
                                # prediction -- the BraTS convention scores that 0.0, or 1.0 when
                                # the reference region is empty, so a dead checkpoint would be
                                # CERTIFIED and reported as merely weak, in silence.  With False
                                # (default) N2 stops and names the cases and the checkpoint; with
                                # True those cases are dropped from the cohort and the report says
                                # how many.  Prefer re-running N1 (it detects nan weights on resume
                                # and retrains that arm).
MIN_CAL     = 8                  # below this many calibration cases the finite-sample
                                 # bound is so wide that the certificate carries no
                                 # information: the notebook says so instead of printing it
MIN_STRATUM_N = 8               # minimum calibration cases per contrast stratum.  Below
                                # this the subgroup bound is uninformative, so the report
                                # falls back to the pooled certificate and says so.


# ---------------------------------------------------------------- build fingerprint
BUILD = "N2 2026-10-09a cell-4 memory fix + per-case curve cache + partial-cache identity check + eval-cache key + skel RNG-stream fix (F1) + best-step/run-budget guard fix (G1)"
print(f"BUILD: {BUILD}")
print(f"  expected fixes in this copy: conformal certificates calibrated on VALIDATION only | post-processing tuned on validation and frozen before TEST | pooled + stratified risk | a `probs` cache must prove WHICH model made it (fingerprint, or a one-case probe)")
print("  If you do NOT see this BUILD line in the output, you are running an older copy of the "
      "notebook: re-import "
      f"N2_eval_certify.ipynb from kaggle_v2_scripts.zip (File -> Import Notebook) instead of hand-patching "
      "cells.  Half of one build and half of another is what makes a fix look like it did not work.")


def _n2_check_1():
    import platform, shutil, subprocess
    print("=" * 88)
    print("CHECK 1/3  environment, memory, disk -- before stage 1")
    print("=" * 88)
    print(f"  python  {platform.python_version()} on {platform.system()} {platform.machine()}")
    try:
        for ln in open("/proc/meminfo"):
            if ln.startswith(("MemTotal", "MemAvailable")):
                print("  ram    ", ln.strip())
    except Exception:
        pass
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                            "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
        print("  gpu    ", (r.stdout.strip() or r.stderr.strip() or "no output"))
    except FileNotFoundError:
        print("  gpu     nvidia-smi not found -- CPU session: inference and calibration will be "
              "very slow")
    for _p in [x for x in {globals().get("WORK"), "/kaggle/working"} if x]:
        try:
            print(f"  disk    {shutil.disk_usage(_p).free / 2**30:6.1f} GB free in {_p}")
        except Exception:
            pass
    _cfg = {k: globals().get(k) for k in ("BUILD", "RUN_TEST", "ARMS", "MIN_CAL", "PROBE_TOL",
                                          "LAMBDA_GRID", "FILTER_ALPHAS")}
    print("  config  " + ", ".join(f"{k}={_cfg[k]!r}" for k in _cfg if _cfg[k] is not None))
    print("  expect  cell 4 -> 'runs: ...' + 'split: ...' + 'val 37 | test 37 | manifest ... rows'")
    print("  expect  cell 4 -> best.pt selected-step fingerprints + 'training completion' totals from DONE/last.pt")
    print("  expect  cell 5 -> a 'model identity:' line per arm (kept / computed / recomputed)")
    print("  expect  cell 7 -> 'calibrating on 37 validation cases (region WT) ...'")
    print("  expect  cell 8 -> 'filter_feasible: false' and 'filter_delta: 1' (structural at "
          "n_cal < 99 -- see N2_REVIEW.md, do not 'fix' it)")
    print("CHECK 1/3 done")


_n2_check_1()


BUILD: N2 2026-10-09a cell-4 memory fix + per-case curve cache + partial-cache identity check + eval-cache key + skel RNG-stream fix (F1) + best-step/run-budget guard fix (G1)
  expected fixes in this copy: conformal certificates calibrated on VALIDATION only | post-processing tuned on validation and frozen before TEST | pooled + stratified risk | a `probs` cache must prove WHICH model made it (fingerprint, or a one-case probe)
  If you do NOT see this BUILD line in the output, you are running an older copy of the notebook: re-import N2_eval_certify.ipynb from kaggle_v2_scripts.zip (File -> Import Notebook) instead of hand-patching cells.  Half of one build and half of another is what makes a fix look like it did not work.
CHECK 1/3  environment, memory, disk -- before stage 1
  python  3.13.15 on Linux x86_64
  ram     MemTotal:       32869452 kB
  ram     MemAvailable:   30422224 kB
  gpu     Tesla T4, 15360 MiB, 3 MiB
Tesla T4, 15360 MiB, 3 MiB
  disk      19.5 GB free in /kaggle/wo

In [18]:
# %% [CELL 3] SHARED LIBRARY (bratsx + safepp) -- identical in every notebook -----
from __future__ import annotations

# ============================================================================
# bratsx.py -- shared library for the BraTS2020 x fractional-DoD study
# (attached to each notebook as a single "SHARED LIBRARY" cell)
#
# Contents
#   A. fractional operators  : riesz_weights, RieszConv3d, FractionalOrderMixture,
#                              spectral_frac_laplacian, gaussian_blur3d
#   B. prior head            : FracPrior (frozen-DoD | learnable-mixture | DoG)
#   C. model zoo             : SegResNetRefine (control), FracPriorNet (arms)
#   D. data pipeline         : decode_*, PatchDataset, sliding-window inference
#   E. metrics               : dice_np, hd95_fast, band_dice, postprocess
#   F. kaggle plumbing       : session budget, run-queue markers, disk guards
# ============================================================================
import glob, hashlib, json, math, os, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MODS = ["flair", "t1", "t1ce", "t2"]
DOD_CH = (0, 2)                      # FLAIR, T1ce -- the modalities the DoD arms consume
FULL_VAL_CASES = 74                  # the fixed split has 37 val / 37 test


# ===========================================================================
# A. fractional operators
# ===========================================================================
def _sgn_gamma(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x > 0, torch.ones_like(x), (-1.0) ** (torch.floor(-x) + 1))


def riesz_weights(alpha: torch.Tensor, K: int = 3) -> torch.Tensor:
    """Riesz / fractional centred-difference taps, differentiable in alpha, zero-sum.
    alpha = 2 -> [1,-2,1].  Returns (C, 2K+1)."""
    a = alpha.reshape(-1).to(torch.float64)
    ks = torch.arange(-K, K + 1, dtype=torch.float64, device=a.device)
    x0, xa, xb = a + 1.0, a[:, None] / 2 - ks[None, :] + 1, a[:, None] / 2 + ks[None, :] + 1
    logabs = torch.lgamma(x0)[:, None] - torch.lgamma(xa) - torch.lgamma(xb)
    sgn = _sgn_gamma(x0)[:, None] * _sgn_gamma(xa) * _sgn_gamma(xb) * (-((-1.0) ** ks))
    w = sgn * torch.exp(logabs)
    pole = ((xa <= 0) & (xa == xa.round())) | ((xb <= 0) & (xb == xb.round()))
    w = torch.where(pole, torch.zeros_like(w), w).clone()
    w[:, K] = w[:, K] - w.sum(dim=1)
    return torch.nan_to_num(w, nan=0.0).to(alpha.dtype)


def riesz_weights_np(alpha: float, K: int = 3):
    """numpy twin, for data-side use (matches the reference implementation)."""
    from scipy.special import gamma, rgamma
    ks = np.arange(-K, K + 1)
    g = (-1.0) ** ks * gamma(alpha + 1) * rgamma(alpha / 2 - ks + 1) * rgamma(alpha / 2 + ks + 1)
    w = -g
    w[K] -= w.sum()
    return w, ks


def _conv1d_axis(x, w, K, dim, groups):
    pad = [0] * 6
    pad[2 * (2 - dim)] = K
    pad[2 * (2 - dim) + 1] = K
    xp = F.pad(x, pad, mode="replicate")
    C = x.shape[1]
    shape = [1, 1, 1, 1, 1]
    shape[2 + dim] = 2 * K + 1
    return F.conv3d(xp, w.reshape(shape).expand(C, 1, *shape[2:]).to(x.dtype), groups=C)


class FractionalOrderMixture(nn.Module):
    """Softmax mixture over a fixed grid of orders.  Returns Delta^alpha x with
    E[alpha] = sum_a p_a alpha_a -- the number to report.  Numerically bulletproof
    (no Gamma poles, unlike a continuous alpha)."""

    def __init__(self, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), K=3, temperature=1.0,
                 init_probs=None):
        super().__init__()
        self.K = K
        self.register_buffer("alphas", torch.tensor(alphas, dtype=torch.float32))
        self.register_buffer("kernels", torch.stack(
            [torch.tensor(riesz_weights_np(float(a), K)[0], dtype=torch.float32) for a in alphas]))
        self.temperature = temperature
        init = torch.ones(len(alphas)) if init_probs is None else torch.tensor(np.asarray(init_probs), dtype=torch.float32)
        self.logits = nn.Parameter(torch.log(init / init.sum()))

    def posterior(self):
        return torch.softmax(self.logits / self.temperature, dim=0)

    def mean_alpha(self):
        return (self.posterior() * self.alphas).sum()

    def forward(self, x):
        w = self.posterior() @ self.kernels
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))

    def extra_repr(self):
        return f"E[alpha]={float(self.mean_alpha()):.3f} p={[round(float(v),3) for v in self.posterior()]}"


class RieszConv3d(nn.Module):
    def __init__(self, channels, K=3, alpha_init=1.5, per_channel=True, lo=0.2, hi=2.0):
        super().__init__()
        self.K, self.channels, self.per_channel, self.lo, self.hi = K, channels, per_channel, lo, hi
        n = channels if per_channel else 1
        u = (alpha_init - lo) / (hi - lo)
        self.alpha_raw = nn.Parameter(torch.full((n,), math.log(u / (1 - u))))

    @property
    def alpha(self):
        a = self.lo + (self.hi - self.lo) * torch.sigmoid(self.alpha_raw)
        return a if self.per_channel else a.expand(self.channels)

    def forward(self, x):
        w = riesz_weights(self.alpha, self.K)[:, None, :].to(x.dtype)     # (C,1,2K+1)
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))


def _frac_symbol(shape, alpha, radial=True, device="cpu"):
    fs = [torch.fft.fftfreq(n, device=device, dtype=torch.float32) * 2 * math.pi for n in shape]
    g = torch.meshgrid(*fs, indexing="ij")
    if radial:
        return -((sum(4.0 * torch.sin(x / 2.0) ** 2 for x in g)) + 1e-12) ** (alpha / 2.0)
    return -sum((4.0 * torch.sin(x / 2.0) ** 2 + 1e-12) ** (alpha / 2.0) for x in g)


def spectral_frac_laplacian(x, alpha, radial=True):
    S = _frac_symbol(x.shape[-3:], alpha, radial, x.device)
    X = torch.fft.fftn(x.float(), dim=(-3, -2, -1))
    return torch.fft.ifftn(X * S, dim=(-3, -2, -1)).real.to(x.dtype)


def gaussian_blur3d(x, sigma, truncate=3.0):
    """separable Gaussian blur, groups=C (the classical anchor for the DoG arm)"""
    if sigma <= 0:
        return x
    r = int(truncate * sigma + 0.5)
    k = torch.exp(-0.5 * (torch.arange(-r, r + 1, device=x.device, dtype=torch.float32) / sigma) ** 2)
    k = (k / k.sum()).to(x.dtype)
    C = x.shape[1]
    out = x
    for d in range(3):
        pad = [0] * 6
        pad[2 * (2 - d)] = r
        pad[2 * (2 - d) + 1] = r
        shape = [1, 1, 1, 1, 1]
        shape[2 + d] = 2 * r + 1
        out = F.conv3d(F.pad(out, pad, mode="replicate"),
                       k.reshape(shape).expand(C, 1, *shape[2:]), groups=C)
    return out


def spectral_dod_volume(x, alpha, T1=4.0, T2=12.0, radial=True, pad=24,
                        signed=False, smooth_sigma=0.0):
    """Exact DoD of one volume, x = (1,1,D,H,W) or (D,H,W) float tensor.
    smooth_sigma > 0 pre-smooths (stand-in for the papers' RBF interpolant).
    Returns |DoD|, or the tuple (|DoD|, signed DoD) when signed=True."""
    u = x.float()[None, None] if x.dim() == 3 else x.float()
    if smooth_sigma and smooth_sigma > 0:
        from scipy.ndimage import gaussian_filter
        u = torch.from_numpy(gaussian_filter(u[0, 0].numpy(), smooth_sigma, mode="nearest"))[None, None]
    up = F.pad(u, [pad] * 6, mode="replicate") if pad else u
    S = _frac_symbol(up.shape[-3:], alpha, radial, "cpu")
    H = (torch.exp(T2 * S) - torch.exp(T1 * S)).float()
    y = torch.fft.ifftn(torch.fft.fftn(up) * H, dim=(-3, -2, -1)).real
    if pad:
        y = y[..., pad:-pad, pad:-pad, pad:-pad]
    return (y.abs(), y) if signed else y.abs()


def match_band_T(alpha, ref_alpha=2.0, ref_T1=4.0, ratio=3.0, radial=True):
    """T1 such that the DoD band centre of `alpha` coincides with the reference
    (alpha=2, T1=4) band centre.  alpha=1.5 -> (2.435, 7.305).  radius=3."""
    k = 2.0 * math.pi / 15.0                      # band centre used by the reference
    ref_sigma = (2.0 * k ** 2) ** (ref_alpha / 2.0) / 2.0 if radial else k ** ref_alpha / 2.0
    sigma = (math.log(ratio) / (ref_T1 * (ratio - 1.0))) if ref_T1 else None
    # numerical solve is more robust than the closed form above
    def peak_w(a, T1):
        ww = np.linspace(1e-4, np.pi, 4000)
        S = -(ww ** a) if not radial else -((2 - 2 * np.cos(ww)) ** (a / 2.0))
        R = np.abs(np.exp(3 * T1 * S) - np.exp(T1 * S))
        return ww[int(np.argmax(R))]
    target = peak_w(ref_alpha, ref_T1)
    lo, hi = 1e-3, 60.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if peak_w(alpha, mid) > target:
            lo = mid
        else:
            hi = mid
    T1 = 0.5 * (lo + hi)
    return float(T1), float(T1 * ratio)


# ===========================================================================
# B. prior head -- the operator's only route into the network
# ===========================================================================
class FracPrior(nn.Module):
    """logits_out = logits_in + sum_m beta[c,m] * R(x)[:, m]

    mode="frozen": R = identity on the precomputed DoD channels (x[:, 4:])
    mode="mix"   : R = Delta^{alpha} with a Learnable Mixture over orders, on x[:, :4]
    mode="dog"   : R = x - G_sigma*x  (classical high-pass anchor), on x[:, :4]

    beta is zero-initialised => at step 0 the arm is BIT-IDENTICAL to the control.
    """

    def __init__(self, n_classes=3, n_ch=4, mode="mix", K=3, alpha_init=1.5,
                 dog_sigma=2.0, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), init_at_alpha2=True):
        super().__init__()
        self.mode, self.n_ch, self.K = mode, n_ch, K
        self.dog_sigma = dog_sigma
        if mode == "mix":
            init = np.ones(len(alphas))
            if init_at_alpha2:
                init[np.argmin([abs(a - 2.0) for a in alphas])] = 4.0   # start near classical
            self.op = FractionalOrderMixture(alphas=alphas, K=K, init_probs=init)
        else:
            self.op = None
        self.beta = nn.Parameter(torch.zeros(n_classes, n_ch))

    def alpha_value(self):
        return float(self.op.mean_alpha()) if self.op is not None else float("nan")

    def operator(self, x):
        mri = x[:, :4]
        if self.mode == "mix":
            return self.op(mri)
        if self.mode == "dog":
            return mri - gaussian_blur3d(mri, self.dog_sigma)
        return x[:, 4:4 + self.n_ch]                     # frozen precomputed DoD

    def forward(self, logits, x):
        r = self.operator(x)
        s = r.flatten(2).std(dim=2).clamp_min(1e-6).view(r.shape[0], -1, 1, 1, 1)
        r = torch.clamp(r / s, -6.0, 6.0)
        return logits + torch.einsum("cm,bm...->bc...", self.beta, r)

    def extra_repr(self):
        return f"mode={self.mode} n_ch={self.n_ch} " + \
               (f"E[alpha]={self.alpha_value():.3f}" if self.op is not None else "")


# ===========================================================================
# C. model zoo
# ===========================================================================
try:                                                  # noqa: E402
    from monai.networks.nets import SegResNet
    from monai.networks.blocks import Convolution
    from monai.losses import DiceLoss
except ImportError as _e:                            # pragma: no cover
    raise ImportError(
        "MONAI is not installed in this image. Fix it in one of two ways, then re-run from the "
        "top: (a) notebook Settings -> Internet = On and run  !pip install monai  in a cell above; "
        "or (b) use the notebook's Dependency Manager to add 'monai'. Kaggle's standard image has "
        "monai, torch, nibabel, scipy, pandas and scikit-learn pre-installed."
    ) from _e


def _segresnet():
    return SegResNet(spatial_dims=3, in_channels=4, out_channels=3, init_filters=16,
                     blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1), dropout_prob=0.2)


class RefineHead(nn.Module):
    """capacity-matched refinement head: sees [coarse probs, input channels]"""

    def __init__(self, in_ch):
        super().__init__()
        last = nn.Conv3d(16, 3, 1)
        nn.init.zeros_(last.weight); nn.init.zeros_(last.bias)
        self.net = nn.Sequential(
            Convolution(spatial_dims=3, in_channels=3 + in_ch, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            Convolution(spatial_dims=3, in_channels=16, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            last)

    def forward(self, coarse, x):
        return self.net(torch.cat([torch.sigmoid(coarse), x], dim=1))


class FracPriorNet(nn.Module):
    """backbone (MRI only) + refine head + optional operator prior.
    mode=None reproduces the capacity-matched control exactly."""

    def __init__(self, in_ch=4, prior_mode=None, alpha_init=1.5, n_prior_ch=4, dog_sigma=2.0):
        super().__init__()
        self.in_ch = in_ch
        self.backbone = _segresnet()
        self.head = RefineHead(in_ch)
        self.prior = None if prior_mode is None else FracPrior(
            n_classes=3, n_ch=n_prior_ch, mode=prior_mode, alpha_init=alpha_init, dog_sigma=dog_sigma)

    def forward(self, x):
        coarse = self.backbone(x[:, :4])
        logits = coarse + self.head(coarse, x)
        if self.prior is not None:
            logits = self.prior(logits, x)
        return logits

    def alpha_value(self):
        return float("nan") if self.prior is None else self.prior.alpha_value()


def build_model(arm: str, alpha=1.5):
    """arm names used across the notebooks"""
    if arm == "mri":
        return _segresnet()                                     # baseline A
    if arm == "mri_ref":
        return FracPriorNet(in_ch=4, prior_mode=None)            # control B (capacity matched)
    if arm == "mri_ref_dod":
        # CONTROL for the extra channels: same 6-ch input, same head, prior switched OFF.
        # dod15 vs mri_ref_dod therefore differs by beta*R ONLY -> bit-identical at init.
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode=None)
    if arm in ("dod15", "dod20", "dod_shuf"):
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode="frozen", n_prior_ch=len(DOD_CH))
    if arm == "dod_fit":
        return FracPriorNet(in_ch=4, prior_mode="mix")
    if arm == "dod_dog":
        return FracPriorNet(in_ch=4, prior_mode="dog")
    raise ValueError(arm)


ALL_ARMS = ("mri", "mri_ref", "mri_ref_dod", "dod15", "dod20", "dod_fit", "dod_dog", "dod_shuf")
DOD_ARMS = ("mri_ref_dod", "dod15", "dod20", "dod_shuf")


def n_input_channels(arm):
    return 4 + (len(DOD_CH) if arm in DOD_ARMS else 0)


# ===========================================================================
# D. data pipeline
# ===========================================================================
def load_nii(path):
    import nibabel as nib
    return np.nan_to_num(nib.load(path).get_fdata(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def list_candidates(pdir, pid, kind, deep=True):
    """Every plausible path for one volume of one case, BEST GUESS FIRST.

    Listing-based and case-INsensitive on purpose.  Real mirrors rename things in ways that
    glob patterns miss: `..._Seg.nii` (capital S), `..._SEG.nii`, `segmentation.nii`,
    `..._seg.nii/..._seg.nii` (the "file" is a folder), a stray suffix, or the bare `seg.nii`.
    A naive `glob(pdir + "*seg*")` returns nothing for the first two, which is what turned a
    single odd case into a dead worker pool (the exception surfaces as IndexError, not as a
    missing-file message).
    """
    exact, base, kind_pid, kind_only = [], [], [], []
    kl, pl = kind.lower(), pid.lower()
    for e in ("nii.gz", "nii", "nii.z", "nii.gz.z", ""):
        nm = f"{pid}_{kind}" + (f".{e}" if e else "")
        for variant in {nm, nm.lower(), nm.upper()}:
            exact.append(os.path.join(pdir, variant))

    def scan(d):
        try:
            entries = sorted(os.listdir(d))
        except OSError:
            return
        for nm in entries:
            full = os.path.join(d, nm)
            if os.path.isdir(full):
                if deep:
                    scan(full)                       # the "file" may be a folder holding it
                continue
            low = nm.lower()
            if low.startswith(pl) and low.startswith(f"{pl}_{kl}"):
                base.append(full)
            elif pl in low and kl in low:
                kind_pid.append(full)
            elif kl in low:
                kind_only.append(full)

    scan(pdir)
    out = []
    for group in (exact, base, kind_pid, kind_only):
        for q in group:
            if os.path.isfile(q) and q not in out:
                out.append(q)
    return out


def case_file(pdir, pid, kind, deep=True):
    """The path to hand to nibabel, or None.  nib.load sniffs the content, so a gzip file
    named `.nii` still loads."""
    cands = list_candidates(pdir, pid, kind, deep=deep)
    return cands[0] if cands else None


def describe_case(pdir, limit=12, kind=None):
    """One-line diagnostic for error messages: what is actually in the case folder (and, for
    the requested volume, the near-misses that were rejected)."""
    try:
        entries = sorted(os.listdir(pdir))
    except OSError as e:
        return f"(cannot list {pdir}: {e})"
    head = entries[:limit]
    more = "" if len(entries) <= limit else f" … +{len(entries) - limit} more"
    txt = f"{pdir} contains {len(entries)} entries: {head}{more}"
    if kind:
        near = [q for q in list_candidates(pdir, pid_of_dir(pdir), kind)
                if os.path.basename(q).lower().find(kind.lower()) >= 0]
        txt += f"\n  names containing {kind!r}: {[os.path.basename(q) for q in near] or 'NONE'}"
        sub = [q for q in sorted(glob.glob(os.path.join(pdir, "**", "*"), recursive=True))
               if os.path.isdir(q)][:5]
        if sub:
            txt += f"\n  subfolders: {[os.path.relpath(q, pdir) for q in sub]}"
    return txt


def pid_of_dir(pdir):
    b = os.path.basename(pdir.rstrip("/"))
    return b if b.startswith("BraTS") else "BraTS20_Training_000"


def find_file(pdir, pid, kind):
    p = case_file(pdir, pid, kind)
    if p is None:
        raise FileNotFoundError(
            f"no '{kind}' volume for {pid}\n  {describe_case(pdir, kind=kind)}\n"
            f"  Accepted: '{pid}_{kind}.nii(.gz)' in any case or spelling, optionally inside a "
            f"folder of the same name, or any file whose name contains '{kind}'.")
    return p


def find_seg(pdir, pid):
    return find_file(pdir, pid, "seg")


def load_case_volumes(pdir, pid, mods=None):
    """(mri_stack, seg) for one case, resolved against whatever the mirror named the files.

    This is what the prep notebook calls: it never constructs a filename by hand, so a
    `.nii`-only mirror, a wrapped-in-folder mirror or an uppercase mirror all work.
    """
    mods = list(mods or MODS)
    vols = [load_nii(find_file(pdir, pid, m)) for m in mods]
    seg = np.rint(load_nii(find_file(pdir, pid, "seg"))).astype(np.uint8)
    seg[seg == 3] = 1                                  # BraTS label convention
    return np.stack(vols), seg


def normalize_mri_single(vol):
    """z-score inside the non-zero mask, clip to the 0.5-99.5 percentile first."""
    m = np.abs(vol) > 1e-6
    out = np.zeros(vol.shape, np.float32)
    if m.sum() < 10:
        return out
    lo, hi = np.percentile(vol[m], [0.5, 99.5])
    v = np.clip(vol[m], lo, hi)
    out[m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def as_roots(root):
    """A cache may be one dir, a LIST of dirs (Kaggle shards), or a glob pattern."""
    if root is None:
        return []
    items = list(root) if isinstance(root, (list, tuple, set)) else [root]
    out = []
    for r in items:
        cands = sorted(glob.glob(str(r))) if any(c in str(r) for c in "*?[") else [str(r)]
        for c in cands:
            if os.path.isdir(c) and c not in out:
                out.append(c)
    return out


def _first_file(roots, name):
    """the first shard that actually contains `name` (loaders never listdir)."""
    for r in roots:
        f = os.path.join(r, name)
        if os.path.exists(f):
            return f
    return os.path.join(roots[0], name) if roots else name


def cache_file(cache, pid, suffix, ext="npy"):
    return _first_file(as_roots(cache), f"{pid}_{suffix}.{ext}")


def has_case(cache, pid):
    return os.path.exists(cache_file(cache, pid, "mri"))


def loader_case(cache, pid, dod_tag=None, dod_root=None):
    """-> (x float32 (C,D,H,W), seg uint8, dist float32).  Every channel is unit scale."""
    c = as_roots(cache)
    dr = as_roots(dod_root) or c
    x = np.load(_first_file(c, f"{pid}_mri.npy")).astype(np.float32)
    if dod_tag:
        d = np.load(_first_file(dr, f"{pid}_{dod_tag}.npy")).astype(np.float32)
        x = np.concatenate([x, d], axis=0)
    return x, np.load(_first_file(c, f"{pid}_seg.npy")), \
        decode_dist(np.load(_first_file(c, f"{pid}_dist.npy")))


def dod_tag(alpha):
    """canonical file tag for a precomputed DoD: 1.5 -> 'dod_a15', 2.0 -> 'dod_a20'"""
    return f"dod_a{int(round(float(alpha) * 10)):02d}"


def decode_dod(u16, range_=32.0):
    return (u16.astype(np.float32) / 65535.0) * range_


def bbox_crop_slices(mri, margin=4):
    brain = (np.abs(mri) > 1e-6).any(axis=0)
    idx = np.where(brain)
    if len(idx[0]) == 0:
        return tuple(slice(0, s) for s in brain.shape)
    return tuple(slice(max(0, int(idx[a].min()) - margin),
                       min(brain.shape[a], int(idx[a].max()) + 1 + margin)) for a in range(3))


def normalize_mri(mri):
    out = np.zeros(mri.shape, dtype=np.float32)
    for c in range(mri.shape[0]):
        m = np.abs(mri[c]) > 1e-6
        if m.sum() < 10:
            continue
        lo, hi = np.percentile(mri[c][m], [0.5, 99.5])
        v = np.clip(mri[c][m], lo, hi)
        out[c][m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def seg_to_targets(seg):
    return np.stack([seg > 0, (seg == 1) | (seg == 4), seg == 4]).astype(np.float32)


def signed_distance_to_wt(seg, clip=40.0):
    from scipy.ndimage import distance_transform_edt
    wt = seg > 0
    if not wt.any():
        return np.zeros(seg.shape, np.float32)
    return np.clip(distance_transform_edt(~wt) - distance_transform_edt(wt), -clip, clip).astype(np.float32)


def encode_dist(d, clip=40.0):
    """signed distance -> uint8 (4.8 MB/case instead of 9.6).  80 voxels / 255 levels
    = 0.31 voxel resolution, which is far finer than the tau=3 weighting needs."""
    z = np.clip((np.asarray(d, np.float32) + clip) / (2 * clip), 0, 1)
    return np.rint(z * 255).astype(np.uint8)


def decode_dist(u, clip=40.0):
    return (np.asarray(u, np.float32) / 255.0) * (2 * clip) - clip


def sample_locs(seg, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    d = {}
    for name, cond in (("c1", seg == 1), ("c2", seg == 2), ("c4", seg == 4)):
        co = np.argwhere(cond)
        if len(co) > n:
            co = co[rng.choice(len(co), n, replace=False)]
        d[name] = co.astype(np.int16)
    return d


class PatchDataset(torch.utils.data.Dataset):
    """Random patch sampler.  Sample i is a pure function of (SEED, offset+i) so every
    arm sees the identical sample sequence -> the comparison is paired by construction.

    Returns (x[C,D,H,W], y[3,D,H,W], dist[1,D,H,W]).  x channels are ALL unit scale.
    """

    P_FG = 0.70

    def __init__(self, cache, ids, arm, length, offset=0, seed=42, patch=(96, 96, 96),
                 dod_tag=None, dod_root=None, shuffle_dod=False, jitter=True):
        self.cache, self.ids, self.arm = as_roots(cache), list(ids), arm
        self.length, self.offset, self.seed, self.patch = length, offset, seed, tuple(patch)
        # dod_tag: which precomputed DoD to load, e.g. "dod_a15" / "dod_a20". None = no DoD.
        # dod_root: the DoD is a SEPARATE dataset from the MRI cache on Kaggle.
        self.dod_tag, self.shuffle_dod, self.jitter = dod_tag, shuffle_dod, jitter
        self.dod_root = as_roots(dod_root) or self.cache
        self.use_dod = dod_tag is not None

    def __len__(self):
        return self.length

    def _p(self, pid, s, ext="npy"):
        return _first_file(self.cache, f"{pid}_{s}.{ext}")

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        # np.array (a copy), not asarray: with a float32 cache the mmap slice is read-only
        # and the jitter write below would raise; the float16 cache up-casts and copies anyway.
        x = np.array(mri[(slice(None),) + sl], dtype=np.float32)
        if self.use_dod:
            src = pid
            if self.shuffle_dod:                      # placebo arm: DoD from ANOTHER case
                src = self.ids[int(rng.integers(len(self.ids)))]
            d = np.load(_first_file(self.dod_root, f"{src}_{self.dod_tag}.npy"), mmap_mode="r")
            d = np.stack([np.asarray(d[c][sl], dtype=np.float32) for c in range(d.shape[0])])
            p = [(0, 0)] + [(0, q - t) for q, t in zip(self.patch, d.shape[1:])]
            if any(b for _, b in p[1:]):
                d = np.pad(d, p)
            x = np.concatenate([x, d], axis=0)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad)
            y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd = np.flip(x, ax), np.flip(y, ax), np.flip(dd, ax)
        if self.jitter:
            for c in range(min(4, x.shape[0])):       # MRI channels only
                if rng.random() < 0.5:
                    m = x[c] != 0
                    x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)))


# ===========================================================================
# E. losses / metrics / inference
# ===========================================================================
def distance_weighted_bce(logits, y, dist, tau=3.0, peak=5.0):
    w = 1.0 + peak * torch.exp(-dist.abs() / tau)
    return F.binary_cross_entropy_with_logits(logits, y, weight=w.expand_as(y), reduction="mean")


def _perimeter_weights(shape, alpha, device):
    fs = [torch.fft.fftfreq(shape[0], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.fftfreq(shape[1], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.rfftfreq(shape[2], device=device, dtype=torch.float32) * 2 * math.pi]
    g = torch.meshgrid(*fs, indexing="ij")
    k = torch.sqrt(sum(x ** 2 for x in g))
    w = (k / (k.max() + 1e-9)) ** alpha
    return w / (w.mean() + 1e-12)


class FractPerimLoss(nn.Module):
    """|k|^alpha seminorm of the probability map, normalised by the target's own
    seminorm -> one lambda is comparable across alpha."""

    def __init__(self, alpha=1.5):
        super().__init__()
        self.alpha = alpha

    def _semi(self, p):
        w = _perimeter_weights(p.shape[-3:], self.alpha, p.device)
        q = p.float() - p.float().mean(dim=(-3, -2, -1), keepdim=True)
        P = torch.fft.rfftn(q, dim=(-3, -2, -1)).abs() ** 2
        return (P * w).sum(dim=(-3, -2, -1)).mean() / float(np.prod(p.shape[-3:]))

    def forward(self, prob, target):
        with torch.no_grad():
            L0 = self._semi(target)
        return self._semi(prob) / (L0 + 1e-8)


class SegLoss(nn.Module):
    def __init__(self, boundary=True, frac_alpha=None, lam_frac=0.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)
        self.boundary = boundary
        self.frac = FractPerimLoss(frac_alpha) if (frac_alpha and lam_frac > 0) else None
        self.lam_frac = lam_frac

    def forward(self, logits, y, dist=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist) if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if self.frac is not None:
            loss = loss + self.lam_frac * self.frac(torch.sigmoid(logits), y)
        return loss


def dice_np(p, t):
    ps, ts = int(p.sum()), int(t.sum())
    return 1.0 if ps + ts == 0 else 2.0 * int((p & t).sum()) / (ps + ts)


def hd95(pred, gt, penalty=373.13):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    ps, gs = bool(pred.any()), bool(gt.any())
    if not ps and not gs:
        return 0.0
    if ps != gs:
        return penalty
    idx = np.where(pred | gt)
    sl = tuple(slice(max(0, int(i.min()) - 1), int(i.max()) + 2) for i in idx)
    p, g = pred[sl], gt[sl]
    sp, sg = p & ~binary_erosion(p), g & ~binary_erosion(g)
    d = np.concatenate([distance_transform_edt(~g)[sp], distance_transform_edt(~p)[sg]])
    return float(np.percentile(d, 95))


def band_mask(mask, r=3):
    from scipy.ndimage import binary_dilation, binary_erosion
    if not mask.any():
        return np.zeros_like(mask)
    return binary_dilation(mask, iterations=r) & ~binary_erosion(mask, iterations=r)


def band_dice(pred, gt, r=3):
    b = band_mask(gt, r)
    return float("nan") if not b.any() else dice_np(pred & b, gt & b)


def clean_components(mask, min_frac=0.1):
    from scipy.ndimage import label
    if not mask.any() or min_frac <= 0:
        return mask
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    keep = sizes >= min_frac * sizes.max(); keep[0] = False
    return keep[lab]


def postprocess(pred, thr=None, cc_wt=0.0, cc_tc=0.0, et_min=0):
    """THE single place where post-processing happens (val, test and tuning all call this)."""
    out = np.zeros_like(pred, bool)
    for c, cc in ((0, cc_wt), (1, cc_tc)):
        m = pred[c] if thr is None else pred[c]
        out[c] = clean_components(m, cc) if cc > 0 else m
    e = pred[2]
    out[2] = e if int(e.sum()) >= et_min else np.zeros_like(e)
    return out


@torch.no_grad()
def infer_probs(model, x_np, patch, overlap, device, amp=True, sw_batch=2, check=True,
                tag=""):
    """Sliding-window probabilities, (3, D, H, W) float16.

    `check=True` refuses to hand back a non-finite probability map.  This matters more than it
    looks: `p >= threshold` with `p = nan` is FALSE everywhere, so a poisoned checkpoint produces
    an EMPTY prediction, which the BraTS convention scores as 0.0 -- or as 1.0 when the reference
    for that region is also empty.  A nan model would therefore be certified and reported as a
    merely weak one, in silence.  Better to stop and name the checkpoint.

    Note that nothing like the training-time "skipped step" can happen here: inference has no
    backward pass, so there is no gradient to be non-finite and no weight to poison.  A non-finite
    probability map can only come from weights that were already nan (an old, pre-scaler
    checkpoint) or from an overflow in the input volumes.
    """
    try:
        from monai.inferers import sliding_window_inference
    except ImportError as _e:                        # pragma: no cover
        raise ImportError("MONAI (monai.inferers) is missing -> !pip install monai, then re-run.") from _e
    model.eval()
    xt = torch.from_numpy(x_np)[None].to(device)
    with torch.amp.autocast("cuda", enabled=amp, dtype=torch.float16):
        o = sliding_window_inference(xt, patch, sw_batch_size=sw_batch, predictor=model,
                                     overlap=overlap, mode="gaussian")
    if check:
        bad_x = int((~torch.isfinite(xt)).sum())
        bad_o = int((~torch.isfinite(o)).sum())
        if bad_o or bad_x:
            _w = max([float(q.abs().max()) for q in model.parameters()], default=0.0)
            raise RuntimeError(
                f"non-finite probabilities: {bad_o} of {o.numel()} logits are nan/inf "
                f"(input volumes: {bad_x} bad voxels; largest weight {_w:.3g})"
                f"{(' for ' + tag) if tag else ''}.\n"
                f"  Inference itself cannot overflow into nan -- there is no backward pass, so "
                f"there is no bad gradient to skip.  This means the WEIGHTS are already nan: the "
                f"checkpoint was written by a run that had already been poisoned (a build without "
                f"the GradScaler).  Do not evaluate or certify it: an all-nan map compares as an "
                f"EMPTY prediction, which the BraTS convention scores 0, or 1 when the reference "
                f"region is empty -- i.e. a dead model would look merely weak.  N1 detects this on "
                f"resume (it prints 'last.pt holds nan weights' and retrains that arm), and "
                f"code/check_run.py flags a run whose loss went non-finite; re-run N1 with "
                f"EXTEND=True (or delete that arm's *.pt) and it will be retrained.")
    return torch.sigmoid(o.float())[0].cpu().numpy().astype(np.float16)



# Bump this whenever the TTA maths changes: N2 folds it into the probability-cache provenance,
# so a cache written by an older implementation is recomputed instead of silently reused.
TTA_IMPL = 2


def infer_probs_tta(model, x_np, patch, overlap, device, flips=(), amp=True, sw_batch=2,
                    tag=""):
    """Flip-averaged sliding-window probabilities: the same model, the same weights, averaged
    over axis flips of the input.

    Why it is free accuracy: a segmentation network is not equivariant to flips, so each flipped
    pass makes a slightly different mistake; averaging the PROBABILITIES (before thresholding,
    where averaging is meaningful) removes part of that variance.  No training, no new weights,
    and the same treatment is applied to every arm, so the arm-to-arm comparison stays fair.

    flips : sequence of axis tuples, e.g. `[(), (0,), (1,), (2,)]` -- the identity plus one flip
            per spatial axis.  Each additional entry costs one more sliding-window pass.
            `()` or `None` means plain inference (no averaging, identical to `infer_probs`).
    """
    combos = [tuple(f) for f in (flips or [()])]
    if len(combos) <= 1:
        return infer_probs(model, x_np, patch, overlap, device, amp=amp, sw_batch=sw_batch,
                           check=True, tag=tag)
    acc = None
    for f in combos:
        # contiguous: np.flip produces negative strides, which torch's from_numpy refuses
        x = (np.ascontiguousarray(np.flip(x_np, axis=tuple(a + 1 for a in f)))
             if f else x_np)                                     # +1: channel axis first
        pr = infer_probs(model, x, patch, overlap, device, amp=amp, sw_batch=sw_batch,
                         check=True, tag=f"{tag} [flip {f or 'none'}]").astype(np.float32)
        if f:
            # `pr` is channel-first (3, D, H, W), exactly like the input -- so undoing a flip
            # along spatial axis `a` means axis `a + 1`.  Flipping `axis=f` here would flip the
            # CHANNEL axis for f=(0,), i.e. swap the WT/TC/ET maps and average three mutually
            # inconsistent volumes; that is a silent collapse of every Dice (observed: 0.84 ->
            # 0.03) rather than an error, which is why it needs a unit test, not a code review.
            pr = np.flip(pr, axis=tuple(a + 1 for a in f))
        acc = pr if acc is None else acc + pr
    out = (acc / len(combos)).astype(np.float16)
    if not np.isfinite(out).all():
        raise RuntimeError(f"flip-averaged probabilities are non-finite{(' for ' + tag) if tag else ''}")
    return out


# ===========================================================================
# F. kaggle plumbing
# ===========================================================================
class SessionBudget:
    """Graceful wall-clock guard: the notebook always exits cleanly and saves its
    checkpoints before the 9 h GPU cap hits."""

    def __init__(self, hours):
        self.t0 = time.time()
        self.limit = hours * 3600

    def left(self):
        return self.limit - (time.time() - self.t0)

    def ok(self, need=0.0):
        return self.left() > need

    def hhmm(self):
        s = max(0, int(self.left()))
        return f"{s//3600}h{(s%3600)//60:02d}m"


def run_done(run_dir):
    return os.path.exists(os.path.join(run_dir, "DONE"))


def mark_done(run_dir, payload=None):
    os.makedirs(run_dir, exist_ok=True)
    if payload is not None:
        json.dump(payload, open(os.path.join(run_dir, "DONE"), "w"), indent=2)
    else:
        open(os.path.join(run_dir, "DONE"), "w").write("ok")


def adopt_previous_runs(src, dst):
    """Copy everything from an attached dataset (previous sessions' output) into the
    working dir so the run queue can skip finished runs.  This is what makes the
    study resumable across Kaggle sessions, which do not persist /kaggle/working."""
    if not src or not os.path.isdir(src):
        print(f"adopt: nothing at {src}")
        return
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        for f in files:
            d = os.path.join(dst, rel, f)
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True)
                shutil.copy2(os.path.join(root, f), d)
                n += 1
    print(f"adopt: {n} new file(s) copied from {src} -> {dst}")


def disk_report(*paths):
    for p in paths:
        if os.path.isdir(p):
            tot = sum(os.path.getsize(os.path.join(r, f))
                      for r, _, fs in os.walk(p) for f in fs) / 1e9
            print(f"  {p:55s} {tot:6.2f} GB")
        elif os.path.exists(p):
            print(f"  {p:55s} {os.path.getsize(p)/1e6:6.1f} MB")
        else:
            print(f"  {p:55s} (missing)")


def peak_gpu_mem_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
"""safepp.py -- "safe post-processing": the mathematical core of the glioma reading aid.

Standalone (numpy + scipy only, no torch, no GPU, no Kaggle).  Everything here is
deterministic, unit-testable, and calibrated on VALIDATION data only.

Contents
--------
A. morphology / post-processing operators
     components()               26-connected component labelling
     filter_components()        remove components below a physical size
     relabel_et_to_necrosis()   nnU-Net BraTS post-processing step
     postproc_seg()             the two operators as one validated pipeline

B. risk control  (Angelopoulos et al., Conformal Risk Control, ICLR 2024)
     miss_curve()               per-case "fraction of tumour missed" vs threshold
     crc_threshold()            lambda-hat with the finite-sample correction
     stratified_crc()           group-conditional calibration per stratum

C. certified component filtering
     removal_curve()            fraction of true lesion volume a filter would delete
     certified_filter_threshold()

D. metrics
     dice_regions(), hd95(), lesionwise_dice(), component_stats(), volume_error()

Every public function has a docstring stating what it guarantees and what it does NOT
guarantee -- read those before quoting a number in a presentation.

Run `python3 safepp.py` for the self-test suite (no arguments, no data needed).
"""

import numpy as np
from scipy import ndimage

# --------------------------------------------------------------------------------------
# A. post-processing operators
# --------------------------------------------------------------------------------------

CONN26 = np.ones((3, 3, 3), dtype=bool)          # 26-connectivity for 3D components

# BraTS label convention used throughout the harness
NCR, ED, ET = 1, 2, 4
REGIONS = {"WT": (1, 2, 4), "TC": (1, 4), "ET": (4,)}


def region_thresholds(lam, params=None):
    """The threshold actually applied to each region's channel.

    WT -> the certified lambda, untouched.  TC/ET -> an optional per-region override from the
    frozen post-processing (tuned on validation like everything else).  The certificate is
    about WT, and the WT mask here is `seg > 0`, which depends only on the WT threshold -- so
    a TC/ET override cannot invalidate it.  That separation is the whole point: one shared
    threshold has to compromise between a large diffuse region and a small one, which is how
    cases end up with an empty ET prediction and a Dice of exactly 0.
    """
    lam = float(lam)
    p = params or {}
    out = {"WT": lam}
    for key, cap in (("thr_tc", "TC"), ("thr_et", "ET")):
        v = p.get(key)
        out[cap] = lam if v is None else float(v)
    return out


def labels_from_probs(p_wt, p_tc, p_et, thr):
    """Nested BraTS labels from three foreground probability maps.

    `thr` is either one number for all three regions or a dict/tuple of per-region thresholds
    (see `region_thresholds`).  Nested by construction: ET sits inside TC sits inside WT, so
    the label map can never contain an ET island outside the tumour.  This is the ONLY place
    where probabilities become a mask -- val, calibration and test all call it, which is what
    makes the certified threshold meaningful.
    """
    if isinstance(thr, dict):
        t_wt = float(thr.get("WT", 0.5))
        t_tc = float(thr.get("TC", t_wt))
        t_et = float(thr.get("ET", t_tc))
    elif isinstance(thr, (tuple, list, np.ndarray)):
        vals = [float(v) for v in thr]
        t_wt = vals[0]
        t_tc = vals[1] if len(vals) > 1 else t_wt
        t_et = vals[2] if len(vals) > 2 else t_tc
    else:
        t_wt = t_tc = t_et = float(thr)
    wt, tc, et = p_wt >= t_wt, p_tc >= t_tc, p_et >= t_et
    seg = np.zeros(p_wt.shape, np.uint8)
    seg[wt] = ED
    seg[wt & tc] = NCR
    seg[wt & tc & et] = ET
    return seg


def components(mask, conn=CONN26):
    """Label 26-connected components of a binary mask.

    Returns (labels int32, n_components).  `labels == 0` is background.
    """
    lab, n = ndimage.label(np.asarray(mask, dtype=bool), structure=conn)
    return lab.astype(np.int32), int(n)


def component_sizes(mask, conn=CONN26):
    """Sorted-descending array of component sizes (voxels).  Empty if mask is empty."""
    lab, n = components(mask, conn)
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    return np.bincount(lab.ravel())[1:].astype(np.int64)


def filter_components(mask, min_voxels, conn=CONN26):
    """Keep only components with >= `min_voxels` voxels.

    Deterministic, idempotent, and monotone: raising the threshold can only remove more
    voxels.  That monotonicity is what makes the certified variant in section C valid.
    """
    mask = np.asarray(mask, dtype=bool)
    if min_voxels <= 1:
        return mask.copy()
    lab, n = components(mask, conn)
    if n == 0:
        return mask.copy()
    keep = np.bincount(lab.ravel(), minlength=n + 1) >= min_voxels
    keep[0] = False
    return keep[lab]


def relabel_et_to_necrosis(seg, min_et_voxels, ncr=NCR, et=ET):
    """nnU-Net's BraTS post-processing: if the enhancing tumour is smaller than
    `min_et_voxels`, relabel it to non-enhancing tumour/necrosis.

    Rationale (Isensee et al., BraTS 2020): tiny ET predictions are usually false
    positives near vessels/choroid plexus, and the challenge metric scores an empty
    prediction in an ET-less case as perfect.  Their paper also warns this *removes*
    true small enhancements, which is exactly why we certify it in section C.
    """
    seg = np.asarray(seg).copy()
    if min_et_voxels is None or min_et_voxels <= 0:
        return seg
    et_mask = seg == et
    if int(et_mask.sum()) < min_et_voxels:
        seg[et_mask] = ncr
    return seg


def postproc_seg(seg, params):
    """Apply the validated post-processing pipeline to a label map.

    params: dict(min_vox={region: int}, min_et_voxels=int, keep_largest=bool)
            `min_vox` is applied per region on the region mask; a voxel is removed only
            if it is not part of *any* kept region, so WT/TC/ET stay nested.
    """
    seg = np.asarray(seg).copy()
    min_vox = (params or {}).get("min_vox") or {}
    # 1. region-wise size filtering, nested: compute keep masks then re-derive labels
    wt_keep = filter_components(np.isin(seg, REGIONS["WT"]), int(min_vox.get("WT", 1)))
    tc_keep = filter_components(np.isin(seg, REGIONS["TC"]), int(min_vox.get("TC", 1)))
    et_keep = filter_components(np.isin(seg, REGIONS["ET"]), int(min_vox.get("ET", 1)))
    if (params or {}).get("keep_largest_wt"):
        lab, n = components(wt_keep)
        if n > 1:
            sizes = np.bincount(lab.ravel(), minlength=n + 1)
            wt_keep = lab == int(np.argmax(sizes[1:]) + 1)
    out = np.zeros_like(seg)
    out[wt_keep] = seg[wt_keep]                     # keep original labels inside WT
    out[~wt_keep] = 0
    et_removed = np.isin(seg, REGIONS["ET"]) & ~et_keep & wt_keep
    out[et_removed] = NCR                           # drop ET but keep it as core
    tc_removed = np.isin(seg, REGIONS["TC"]) & ~tc_keep
    out[tc_removed & (out != 0)] = ED               # drop TC -> falls back to edema
    # 2. ET relabeling last, so it sees the cleaned ET
    out = relabel_et_to_necrosis(out, (params or {}).get("min_et_voxels", 0))
    return out


# --------------------------------------------------------------------------------------
# B. conformal risk control
# --------------------------------------------------------------------------------------

def _as_prob(p):
    p = np.asarray(p, dtype=np.float32)
    return p


def miss_curve(prob, gt_fg, thresholds):
    """Per-threshold "fraction of ground-truth tumour voxels missed".

    prob    : (D,H,W) foreground probability
    gt_fg   : (D,H,W) boolean ground truth
    returns : array of shape (len(thresholds),) -- risk for "predict tumour if p >= t"

    Risk is non-decreasing in the threshold t (a stricter threshold recalls less), and
    bounded in [0, 1].  Empty ground truth -> risk 0 (nothing can be missed).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    n_fg = int(gt_fg.sum())
    if n_fg == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    v = prob[gt_fg]
    return np.array([float((v < t).mean()) for t in thresholds], dtype=np.float64)


def crc_threshold(calib_risks, alpha, B=1.0, thresholds=None):
    """Conformal risk control threshold with the finite-sample correction.

    calib_risks : (n_cal, n_thresholds) per-calibration-case risk, non-decreasing in the
                  threshold axis (as produced by `miss_curve`).
    alpha       : target risk level in (0, 1)
    B           : upper bound on the loss (1.0 for a fraction)

    Returns dict(threshold, lam, feasible, mean_risk_cal, bound_cal, risk_curve,
                 thresholds) where the guarantee is

        E[ risk(new case, threshold) ] <= alpha      (expectation over the calibration draw)

    Why the correction: with a finite calibration set the plain empirical quantile is
    optimistic; CRC adds B/(n+1) and rescales by n/(n+1), which yields the bound above
    with no distributional assumptions (Angelopoulos et al. 2024, Theorem 1).

    What it does NOT give: per-case coverage, or coverage inside a subgroup -- that is
    `stratified_crc`.  If no threshold satisfies the bound the function returns the most
    conservative threshold it has (t = min grid, i.e. the largest prediction set) with
    feasible=False: report that the model cannot meet this risk level, do not pretend.
    """
    R = np.asarray(calib_risks, dtype=np.float64)
    n = R.shape[0]
    if n == 0:
        raise ValueError("empty calibration set")
    if thresholds is None:
        thresholds = np.linspace(0, 1, R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"risk array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        # Tightest set that still meets the target: risk is non-decreasing in the
        # threshold, so the LARGEST feasible threshold is the smallest prediction set
        # that keeps the guarantee.  Taking ok[0] instead would return "predict
        # everything", which is a vacuous certificate.
        j = int(ok[-1])
        feasible = True
    else:
        j, feasible = 0, False            # no threshold meets alpha -> say so, don't fake it
    return dict(threshold=float(thresholds[j]), lam=float(1.0 - thresholds[j]),
                feasible=feasible, mean_risk_cal=float(mean_risk[j]),
                bound_cal=float(bound[j]), risk_curve=mean_risk, bound_curve=bound,
                thresholds=thresholds, n_cal=n)


def stratified_crc(calib_risks_by_group, alpha, B=1.0, thresholds=None):
    """Group-conditional CRC: one threshold per stratum.

    Group-conditional calibration gives E[risk | group] <= alpha for each group, instead
    of only E[risk] <= alpha overall.  The statistical price is a smaller calibration set
    per group (so a looser, larger prediction set) -- report that price, it is the honest
    part of the claim.

    returns {group: crc_threshold(...)} and, for convenience, the pooled threshold under
    key "__pooled__".
    """
    out = stratified_crc.__wrapped__ if False else {}
    pooled = np.concatenate([np.asarray(v) for v in calib_risks_by_group.values()], axis=0)
    out["__pooled__"] = crc_threshold(pooled, alpha, B=B, thresholds=thresholds)
    for g, R in calib_risks_by_group.items():
        out[g] = crc_threshold(np.asarray(R), alpha, B=B, thresholds=thresholds)
    return out


def fp_curve(prob, gt_fg, thresholds):
    r"""Fraction of the PREDICTION that is wrong: |pred_theta \ GT| / |pred_theta|.

    This is the over-segmentation counterpart of `miss_curve`: it FALLS as the threshold
    rises (a stricter threshold throws away the least confident -- usually spurious --
    voxels first).  Empty prediction -> 0 by convention (nothing predicted, nothing wrong).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    out = []
    for t in thresholds:
        pred = prob >= t
        n = int(pred.sum())
        out.append(0.0 if n == 0 else float((pred & ~gt_fg).sum()) / n)
    return np.array(out, dtype=np.float64)


def monotone_envelope(r, direction="increasing"):
    """Replace a noisy risk curve by its worst-case monotone envelope.

    Certifying the envelope certifies the deployed operator, because at every threshold
    the envelope is >= the realised risk:
      * increasing family -> running MAXIMUM over smaller thresholds
      * decreasing family -> running MAXIMUM over larger thresholds
    The alternative (`minimum.accumulate`) looks tighter and is WRONG: it would certify a
    risk lower than the operator actually carries.
    """
    r = np.asarray(r, dtype=np.float64)
    # ALONG THE THRESHOLD AXIS ONLY.  Calling np.maximum.accumulate without `axis` on a
    # (n_cal, n_thresholds) matrix flattens it in C order, so the running maximum leaks
    # from one calibration case into the next and the bounds inflate by a large factor
    # (observed: mean risk 0.055 -> envelope 0.34), which makes every certificate
    # infeasible and the deployed threshold needlessly conservative.
    if direction == "increasing":
        return np.maximum.accumulate(r, axis=-1)
    return np.maximum.accumulate(r[..., ::-1], axis=-1)[..., ::-1]


def certified_threshold_pair(calib_fn, calib_fp, thresholds, alpha, beta, B=1.0):
    """Two one-sided certificates -> one admissible threshold band.

    calib_fn : (n_cal, K) per-case MISSED-tumour fraction, increasing in the threshold
    calib_fp : (n_cal, K) per-case WRONG-prediction fraction, decreasing in the threshold
    alpha    : bound on the missed fraction      (under-segmentation risk)
    beta     : bound on the wrong-prediction fraction (over-segmentation risk)

    Rule (pre-registered, decided before looking at test): take the tightest threshold
    that meets the missed-tumour target, and accept it only if it also meets the
    wrong-prediction target.  If the two targets cannot be met together at this calibration
    size, `feasible` is False and you report that instead of a number.
    """
    F = monotone_envelope(np.asarray(calib_fn), "increasing")
    G = monotone_envelope(np.asarray(calib_fp), "decreasing")
    assert F.shape == G.shape, "risk arrays must have the same shape"
    n = F.shape[0]
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert F.shape[1] == len(thresholds), "risk arrays and thresholds disagree"
    fn_bound = (n / (n + 1.0)) * F.mean(axis=0) + B / (n + 1.0)
    fp_bound = (n / (n + 1.0)) * G.mean(axis=0) + B / (n + 1.0)
    ok_fn = np.where(fn_bound <= alpha)[0]        # prefix: risk grows with the threshold
    ok_fp = np.where(fp_bound <= beta)[0]         # suffix: risk falls with the threshold
    j_fn = int(ok_fn[-1]) if len(ok_fn) else None
    j_fp = int(ok_fp[0]) if len(ok_fp) else None
    feasible = (j_fn is not None) and (j_fp is not None) and (j_fn >= j_fp)
    j = j_fn if j_fn is not None else 0
    return dict(lam=float(thresholds[j]), feasible=bool(feasible),
                thr_fn=float(thresholds[j_fn]) if j_fn is not None else None,
                thr_fp=float(thresholds[j_fp]) if j_fp is not None else None,
                fn_bound_curve=fn_bound, fp_bound_curve=fp_bound,
                fn_mean_curve=F.mean(axis=0), fp_mean_curve=G.mean(axis=0),
                thresholds=thresholds, alpha=float(alpha), beta=float(beta), n_cal=n)


def choose_certificate(cert, cnr=None, stratified=True):
    """Which certificate should be presented for ONE case?

    The pooled certificate is valid on average over the cohort; a stratified one is valid
    within its subgroup and is therefore the right thing to show when it exists, is feasible,
    and we know the case's contrast.  Falling back to the pooled certificate is silently safe:
    the caller never gets a number out of a stratum whose certificate was infeasible.

    cert : the per-arm entry of certificates.json
    cnr  : the case's contrast-to-noise (wt_cnr); None/NaN -> pooled
    Returns dict(scope, lam, n_cal, curves, pooled_lam, cut) with scope 'low'|'high'|'pooled'.
    """
    strata = (cert or {}).get("strata") or None
    pooled = dict(scope="pooled", lam=(cert or {}).get("lam"), n_cal=(cert or {}).get("n_cal"),
                  curves=(cert or {}).get("curves"), pooled_lam=(cert or {}).get("lam"),
                  cut=(strata or {}).get("cut"))
    if not (stratified and strata) or cnr is None or not np.isfinite(float(cnr)):
        return pooled
    low = float(cnr) <= float(strata["cut"])
    lam = strata.get("lam_low" if low else "lam_high")
    feas = bool(strata.get("feasible_low" if low else "feasible_high"))
    if not (feas and lam is not None):
        return pooled                                   # stratum not certifiable -> pooled
    cur = strata.get("curves_low" if low else "curves_high") or pooled["curves"]
    return dict(scope=("low" if low else "high"), lam=float(lam),
                n_cal=int(strata["n_low" if low else "n_high"]), curves=cur,
                pooled_lam=(cert or {}).get("lam"), cut=float(strata["cut"]))


def volume_interval(v_pred, alpha, beta):
    """Worst-case volume interval implied by the two certified risk levels.

    If a case's missed fraction is at most alpha and its wrong-prediction fraction at most
    beta, then
        true >= (1 - beta) * predicted      (at least that share of the prediction is real)
        true <= predicted / (1 - alpha)     (the missed tumour cannot exceed that share)
    so the interval is guaranteed *given* those per-case risk levels.

    IMPORTANT: the certificates bound the AVERAGE risk over cases, not each individual
    case, so this is the worst-case consequence for a case sitting at the certified level.
    Always report the empirical containment rate on held-out cases next to it.
    """
    hi = float(v_pred) / max(1e-9, (1.0 - float(alpha)))
    lo = float(v_pred) * (1.0 - float(beta))
    return (min(lo, hi), max(lo, hi))


def volume_at(prob, gt_fg, threshold, voxel_vol_mm3=1.0):
    """Predicted volume (mm^3) at a threshold, with the ground-truth volume alongside."""
    prob = _as_prob(prob)
    pred = prob >= threshold
    return (float(pred.sum()) * voxel_vol_mm3,
            float(np.asarray(gt_fg, bool).sum()) * voxel_vol_mm3)


def per_case_risk(prob, gt_fg, threshold):
    """Realised risk for one case at one threshold (use this to audit a certificate)."""
    return float(miss_curve(prob, gt_fg, [threshold])[0])


# --------------------------------------------------------------------------------------
# C. certified component filtering
# --------------------------------------------------------------------------------------

def removal_curve(pred_mask, gt_seg, thresholds, region="WT", voxel_vol_mm3=1.0):
    """Per-threshold fraction of ground-truth lesion volume that filtering would delete.

    For each threshold d (in voxels): apply `filter_components` to the predicted region
    mask and measure the ground-truth voxels that the FILTER deleted -- i.e.
    (prediction before AND NOT after) AND ground truth, divided by the ground-truth
    volume.  The result is the filter's collateral damage, per case, in [0, 1].

    Note the subtlety (it changes the clinical claim): voxels the model never predicted
    are NOT charged to the filter, otherwise model error would masquerade as filter risk.

    Non-decreasing in d (see `filter_components` monotonicity), which is what makes the
    CRC bound below valid.
    """
    pred = np.isin(pred_mask, REGIONS[region]) if pred_mask.dtype != bool else pred_mask
    gt = np.isin(gt_seg, REGIONS[region])
    n_gt = int(gt.sum())
    if n_gt == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    out = []
    for d in thresholds:
        kept = filter_components(pred, int(d))
        removed = pred & ~kept                       # what the filter took away
        out.append(float((removed & gt).sum()) / n_gt)
    return np.array(out, dtype=np.float64)


def certified_filter_threshold(calib_removal, alpha, B=1.0, thresholds=None,
                              component_size_floor=None):
    """Largest component-size threshold whose collateral damage is risk-controlled.

    calib_removal : (n_cal, n_thresholds) from `removal_curve`, non-decreasing in d.
    Guarantee: E[deleted true-lesion volume fraction] <= alpha for a new case.
    component_size_floor : optional cap (voxels) on the returned threshold.  Needed
        because the guarantee is EXPECTATIONAL: if the calibration set happens to
        contain no true lesion between 60 and 1600 voxels, the damage curve is flat
        there and the certified threshold can land just below the first big lesion --
        statistically valid, clinically unnerving.  Set the floor from the observed
        distribution of true component sizes on validation (e.g. its 25th percentile)
        and report both numbers: "certified delta = X, engineering floor = Y".

    Because `filter_components` is monotone, the family of operators is nested, so the
    same CRC correction applies (index the family by -d, or equivalently search for the
    LARGEST d that satisfies the bound -- done below).
    """
    R = np.asarray(calib_removal, dtype=np.float64)
    n = R.shape[0]
    if thresholds is None:
        thresholds = np.arange(R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"removal array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        j = int(ok[-1])                   # LARGEST threshold that is still safe
        feasible = True
    else:
        j, feasible = 0, False
    delta, capped = float(thresholds[j]), False
    if component_size_floor is not None and delta > float(component_size_floor):
        delta, capped = float(component_size_floor), True
    return dict(threshold=delta, feasible=feasible, capped=capped,
                certified_threshold=float(thresholds[j]),
                mean_risk_cal=float(mean_risk[j]), bound_cal=float(bound[j]),
                risk_curve=mean_risk, bound_curve=bound, thresholds=thresholds, n_cal=n)


# --------------------------------------------------------------------------------------
# D. metrics
# --------------------------------------------------------------------------------------

def dice(a, b, eps=1e-8):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    return float(2.0 * (a & b).sum() / (a.sum() + b.sum() + eps))


def dice_regions(pred_seg, gt_seg):
    """{WT, TC, ET} voxel-wise Dice.  Empty-prediction convention: if both are empty the
    Dice is defined as 1.0 (the BraTS convention used in the nnU-Net paper)."""
    out = {}
    for r, labs in REGIONS.items():
        p, g = np.isin(pred_seg, labs), np.isin(gt_seg, labs)
        if p.sum() == 0 and g.sum() == 0:
            out[r] = 1.0
        else:
            out[r] = dice(p, g)
    return out


def _surf_dist(a, b, spacing=(1.0, 1.0, 1.0)):
    """Symmetric surface distances: for each surface voxel of a, distance to b (EDT)."""
    if a.sum() == 0 or b.sum() == 0:
        return None
    dt_b = ndimage.distance_transform_edt(~b, sampling=spacing)
    dt_a = ndimage.distance_transform_edt(~a, sampling=spacing)
    sa = a & ~ndimage.binary_erosion(a)          # 6-connectivity surface
    sb = b & ~ndimage.binary_erosion(b)
    if sa.sum() == 0 or sb.sum() == 0:
        return None
    d_ab, d_ba = dt_b[sa], dt_a[sb]
    return np.concatenate([d_ab, d_ba])


def hd95(pred_seg, gt_seg, region="WT", spacing=(1.0, 1.0, 1.0)):
    """95th percentile of the symmetric surface distance, in mm (NaN if undefined).

    Convention: if exactly one side is empty the HD95 is NaN -- do not silently score it
    as 0 or as a huge number; the caller decides (report the count of such cases).
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    d = _surf_dist(p, g, spacing)
    return float("nan") if d is None else float(np.percentile(d, 95))


def lesionwise_dice(pred_seg, gt_seg, region="WT", min_gt_voxels=1, bbox_dilate=0,
                     gt_labels=None):
    """Lesion-wise Dice, averaged over ground-truth connected components of `region`.

    Convention (stated because implementations differ): the ground truth is partitioned
    into 26-connected components; for each component the prediction is restricted to that
    component's bounding box (optionally dilated by `bbox_dilate` voxels); a component
    with no predicted voxels at all scores 0.  Components smaller than `min_gt_voxels`
    are ignored.  Returns (mean, per_lesion_array); NaN if the ground truth has none.
    """
    g = np.isin(gt_seg, REGIONS[region])
    p = np.isin(pred_seg, REGIONS[region])
    # `gt_labels=(lab, n)` from `components(g)` lets a parameter sweep label the ground truth
    # ONCE instead of once per candidate: that is the difference between a few seconds and a
    # few minutes when a tuning grid calls this thousands of times, and it is the same answer.
    lab, n = gt_labels if gt_labels is not None else components(g)
    if n == 0:
        return float("nan"), np.zeros(0)
    scores = []
    for i in range(1, n + 1):
        comp = lab == i
        if comp.sum() < min_gt_voxels:
            continue
        sl = ndimage.find_objects(comp.astype(np.int32))[0]
        sl = tuple(slice(max(0, s.start - bbox_dilate), s.stop + bbox_dilate) for s in sl)
        pc = p[sl]
        scores.append(dice(pc.any() and (pc & comp[sl]), comp[sl]) if pc.any() else 0.0)
    scores = np.array(scores, dtype=np.float64)
    return (float(scores.mean()) if len(scores) else float("nan")), scores


def component_stats(pred_seg, gt_seg, region="WT"):
    """Component bookkeeping for the fragmentation story.

    returns dict(n_pred, n_gt, beta0_error=|n_pred-n_gt|, false_components, missed_lesions,
                 pred_voxels, gt_voxels)
    false_components : predicted components containing no ground-truth voxels
    missed_lesions   : ground-truth components containing no predicted voxels
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    lp, npc = components(p)
    lg, ngc = components(g)
    false_c = 0
    for i in range(1, npc + 1):
        if not (g & (lp == i)).any():
            false_c += 1
    missed = 0
    for i in range(1, ngc + 1):
        if not (p & (lg == i)).any():
            missed += 1
    return dict(n_pred=npc, n_gt=ngc, beta0_error=abs(npc - ngc), false_components=false_c,
                missed_lesions=missed, pred_voxels=int(p.sum()), gt_voxels=int(g.sum()))


def volume_error(pred_seg, gt_seg, region="WT", voxel_vol_mm3=1.0):
    """(predicted - true) volume in mm^3 for one region of one case."""
    p = int(np.isin(pred_seg, REGIONS[region]).sum())
    g = int(np.isin(gt_seg, REGIONS[region]).sum())
    return float((p - g) * voxel_vol_mm3)


def bland_altman(errors):
    """Mean error and 95 % limits of agreement for an array of volume errors."""
    e = np.asarray(errors, dtype=np.float64)
    e = e[~np.isnan(e)]
    if len(e) == 0:
        return dict(n=0, bias=float("nan"), sd=float("nan"), lo=float("nan"), hi=float("nan"))
    bias, sd = float(e.mean()), float(e.std(ddof=1)) if len(e) > 1 else 0.0
    return dict(n=len(e), bias=bias, sd=sd, lo=bias - 1.96 * sd, hi=bias + 1.96 * sd)


# --------------------------------------------------------------------------------------
# self-tests -- no data, no GPU; run this file directly
# --------------------------------------------------------------------------------------

def _toy_case(seed=0, n=48):
    """A synthetic case: one big blobby lesion + optional small satellites."""
    rng = np.random.default_rng(seed)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n].astype(np.float32)
    c = (n - 1) / 2.0
    big = (((xx - c) / 9) ** 2 + ((yy - c) / 7) ** 2 + ((zz - c) / 6) ** 2) <= 1
    small = (((xx - c - 15) / 2.5) ** 2 + ((yy - c + 12) / 2.5) ** 2 + ((zz - c) / 2.5) ** 2) <= 1
    tiny = (((xx - c + 17) / 1.5) ** 2 + ((yy - c - 14) / 1.5) ** 2 + ((zz - c + 8) / 1.5) ** 2) <= 1
    return big, small, tiny


def _test_postproc_operators():
    big, small, tiny = _toy_case()
    assert component_sizes(small)[0] > component_sizes(tiny)[0]
    # 1. filter removes exactly the small components, keeps the big one
    pred = (big | small | tiny)
    kept = filter_components(pred, int(component_sizes(small)[0]) + 1)
    assert np.array_equal(kept, big), "filter must keep only components >= threshold"
    # 2. monotonicity: raising the threshold never adds voxels
    for t in range(1, 400, 37):
        assert (filter_components(pred, t + 10) & ~filter_components(pred, t)).sum() == 0
    # 3. nested-region post-processing: the WT region must survive untouched, and a
    #    removed ET component must fall back to necrosis (never to a spurious label)
    seg = np.zeros_like(big, dtype=np.uint8)
    seg[big] = ED
    seg[small] = ET
    seg[tiny] = ET
    out = postproc_seg(seg, dict(min_vox={"WT": 1, "TC": 1,
                                          "ET": int(component_sizes(tiny)[0]) + 1},
                                 min_et_voxels=0))
    assert set(np.unique(out).tolist()) <= {0, NCR, ED, ET}, np.unique(out)
    assert np.isin(out, REGIONS["WT"]).sum() == np.isin(seg, REGIONS["WT"]).sum(), \
        "WT must be preserved when its own threshold is 1"
    assert (out[tiny] == NCR).all(), "the removed ET component must become necrosis"
    # 4. ET relabeling
    seg2 = np.zeros_like(big, dtype=np.uint8)
    seg2[big] = ED
    seg2[tiny] = ET
    out2 = relabel_et_to_necrosis(seg2, min_et_voxels=int(tiny.sum()) + 1)
    assert (out2 == ET).sum() == 0 and (out2 == NCR).sum() == int(tiny.sum()), "ET must become NCR"
    out3 = relabel_et_to_necrosis(seg2, min_et_voxels=1)
    assert (out3 == ET).sum() == int(tiny.sum()), "a large-enough ET must be kept"
    # 5. empty mask is handled everywhere
    z = np.zeros((8, 8, 8), bool)
    assert filter_components(z, 5).sum() == 0 and component_sizes(z).size == 0
    print("  [ok] post-processing operators (filter, monotonicity, nesting, ET relabel, empty)")


def _toy_probs(seed, n=28, quality=0.85):
    """GT = big|small; predicted probability = quality inside GT, small noise outside."""
    rng = np.random.default_rng(1000 + seed)
    big, small, _ = _toy_case(seed, n)
    gt = big | small
    p = rng.uniform(0.0, 0.35, size=(n, n, n)).astype(np.float32)
    p[gt] = np.clip(quality + rng.normal(0, 0.18, gt.sum()), 0, 1).astype(np.float32)
    return p, gt


def _test_crc_guarantee(n_splits=200, alpha=0.10, n_cal=37, n_test=37, pool_n=300):
    """The headline statistical property: averaged over calibration draws, the realised
    risk of the chosen threshold must not exceed alpha, and the certificate must be
    TIGHT (not the vacuous 'predict everything' solution)."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s) for s in range(pool_n)]
    rng = np.random.default_rng(0)
    realised, thresholds = [], []
    for k in range(n_splits):
        idx = rng.permutation(pool_n)
        cal = [pool[i] for i in idx[:n_cal]]
        tst = [pool[i] for i in idx[n_cal:n_cal + n_test]]
        R = np.stack([miss_curve(p, g, THR) for p, g in cal])
        d = crc_threshold(R, alpha=alpha, thresholds=THR)
        assert d["feasible"], "synthetic model should easily meet alpha=0.10"
        assert d["threshold"] > 0.05, "certificate must not degenerate to predict-everything"
        realised.append(np.mean([per_case_risk(p, g, d["threshold"]) for p, g in tst]))
        thresholds.append(d["threshold"])
    m = float(np.mean(realised))
    assert m <= alpha * 1.02, f"CRC violated: mean realised risk {m:.4f} > {alpha}"
    assert m >= alpha * 0.3, f"suspiciously conservative ({m:.4f}) -- check the formula"
    thr_sd = float(np.std(thresholds))
    print(f"       thresholds chosen: mean {np.mean(thresholds):.3f}, sd {thr_sd:.3f} "
          f"(calibration noise at n_cal={n_cal})")
    # alpha must actually matter: a tighter target yields a smaller realised risk
    R = np.stack([miss_curve(p, g, THR) for p, g in pool[:n_cal]])
    t_lo = crc_threshold(R, 0.05, thresholds=THR)["threshold"]
    t_hi = crc_threshold(R, 0.30, thresholds=THR)["threshold"]
    assert t_lo < t_hi, "smaller alpha must give a larger prediction set (lower threshold)"
    print(f"  [ok] CRC guarantee: mean realised risk {m:.4f} <= alpha {alpha} over "
          f"{n_splits} calibration draws (n_cal={n_cal}); alpha monotone")


def _test_envelope_axis():
    """Regression test for the axis bug described in `monotone_envelope`."""
    rows = np.array([[0.10, 0.20, 0.30],
                     [0.30, 0.10, 0.20]])          # second row has a dip
    up = monotone_envelope(rows, "increasing")
    assert np.allclose(up[0], [0.10, 0.20, 0.30]), up
    assert np.allclose(up[1], [0.30, 0.30, 0.30]), up
    assert up.max() <= 0.30 + 1e-12, "the envelope must not leak across calibration cases"
    down = monotone_envelope(rows, "decreasing")
    assert np.allclose(down[0], [0.30, 0.30, 0.30]) and np.allclose(down[1], [0.30, 0.20, 0.20])
    one_d = monotone_envelope(np.array([0.5, 0.2, 0.4]))
    assert np.allclose(one_d, [0.5, 0.5, 0.5]), one_d
    print("  [ok] monotone envelope acts on the threshold axis only (no cross-case leakage)")


def _test_volume_interval():
    """The two-sided certificate: both risks must hold on held-out data, and the implied
    volume interval must contain the truth for most cases."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s, quality=0.75) for s in range(200)]
    rng = np.random.default_rng(7)
    idx = rng.permutation(200)
    cal_i, tst_i = idx[:60], idx[60:160]
    FN = np.stack([monotone_envelope(miss_curve(pool[i][0], pool[i][1], THR)) for i in cal_i])
    FP = np.stack([monotone_envelope(fp_curve(pool[i][0], pool[i][1], THR), "decreasing")
                   for i in cal_i])
    for alpha, beta in ((0.05, 0.20), (0.10, 0.20), (0.15, 0.30)):
        d = certified_threshold_pair(FN, FP, THR, alpha=alpha, beta=beta)
        assert d["feasible"], f"targets {alpha}/{beta} should be reachable on this toy data"
        assert d["thr_fn"] >= d["thr_fp"], "the admissible band must be a real interval"
        r_fn, r_fp, inside = [], [], []
        for i in tst_i:
            p, g = pool[i]
            pred = p >= d["lam"]
            r_fn.append(float((g & ~pred).sum()) / max(1, int(g.sum())))
            r_fp.append(0.0 if pred.sum() == 0 else float((pred & ~g).sum()) / int(pred.sum()))
            lo, hi = volume_interval(float(pred.sum()), alpha, beta)
            inside.append(lo <= float(g.sum()) <= hi)
        m_fn, m_fp = float(np.mean(r_fn)), float(np.mean(r_fp))
        assert m_fn <= alpha + 0.03, f"missed-tumour certificate violated: {m_fn:.3f} > {alpha}"
        assert m_fp <= beta + 0.05, f"wrong-prediction certificate violated: {m_fp:.3f} > {beta}"
        cov = float(np.mean(inside))
        assert cov >= 0.75, f"volume interval contains the truth only {cov:.0%} of the time"
        print(f"       alpha={alpha:.2f} beta={beta:.2f} -> lambda*={d['lam']:.2f} "
              f"(band [{d['thr_fp']:.2f},{d['thr_fn']:.2f}]), realised FN {m_fn:.3f}, "
              f"FP {m_fp:.3f}, interval containment {cov:.0%}")
    # the two targets move the band in OPPOSITE directions -- that is the whole point:
    #   * a tighter missed-tumour target needs an EARLIER (lower) threshold -> bigger set
    #   * a tighter wrong-prediction target raises the FLOOR of the band
    d_a_lo = certified_threshold_pair(FN, FP, THR, alpha=0.05, beta=0.20)
    d_a_hi = certified_threshold_pair(FN, FP, THR, alpha=0.30, beta=0.20)
    assert d_a_lo["thr_fn"] <= d_a_hi["thr_fn"], (d_a_lo["thr_fn"], d_a_hi["thr_fn"])
    d_b_lo = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.05)
    d_b_hi = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.40)
    assert d_b_lo["thr_fp"] >= d_b_hi["thr_fp"], (d_b_lo["thr_fp"], d_b_hi["thr_fp"])
    print(f"       monotone: alpha 0.05->0.30 moves thr_fn {d_a_lo['thr_fn']:.2f}->"
          f"{d_a_hi['thr_fn']:.2f}; beta 0.05->0.40 moves thr_fp {d_b_lo['thr_fp']:.2f}->"
          f"{d_b_hi['thr_fp']:.2f}")
    print("  [ok] two-sided certificates + volume interval (both risks hold on held-out cases)")


def _test_stratified():
    """Low-quality cases must get their own (more conservative) threshold."""
    THR = np.linspace(0, 1, 201)
    easy = [_toy_probs(s, quality=0.95) for s in range(30)]
    hard = [_toy_probs(s + 500, quality=0.60) for s in range(30)]
    R_easy = np.stack([miss_curve(p, g, THR) for p, g in easy])
    R_hard = np.stack([miss_curve(p, g, THR) for p, g in hard])
    res = stratified_crc({"low_contrast": R_hard, "high_contrast": R_easy}, alpha=0.10,
                         thresholds=THR)
    assert res["low_contrast"]["threshold"] < res["high_contrast"]["threshold"], \
        "the harder stratum must receive the lower (more inclusive) threshold"
    pooled = res["__pooled__"]["threshold"]
    assert res["low_contrast"]["threshold"] <= pooled, "pooling hides the hard stratum"
    print(f"  [ok] stratified CRC: hard stratum threshold {res['low_contrast']['threshold']:.2f} "
          f"< easy {res['high_contrast']['threshold']:.2f} (pooled {pooled:.2f})")


def _test_certified_filter():
    """A filter that deletes true satellites must be bounded by alpha, and the bound must
    stop it from ever deleting the essential (large) lesion."""
    big_size = int(_toy_case(0)[0].sum())
    THR = np.array([1, 10, 25, 50, 100, 200, 400, 800, 1600, 2400])
    cal, tst = [], []
    for s in range(60):
        big, small, tiny = _toy_case(s)
        gt = (big | small | tiny).astype(np.uint8)
        pred = (big | small | tiny).astype(np.uint8)
        if s % 3 == 0:                        # the model sometimes misses the satellite
            pred[small] = 0
        (cal if s < 40 else tst).append(removal_curve(pred, gt, THR, region="WT"))
    d = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR)
    assert d["feasible"], "a modest filter must be certifiable at alpha=0.10"
    j = int(np.where(THR == d["threshold"])[0][0])
    realised = float(np.mean([r[j] for r in tst]))
    # 1. the guarantee holds on held-out cases
    assert realised <= 0.10 + 1e-9, realised
    # 2. the certificate protects essential anatomy: it must never pick a threshold that
    #    would delete the big lesion
    assert d["threshold"] < big_size, (d["threshold"], big_size)
    # 3. a naive fixed threshold is NOT certified -> the certificate added something
    RC = np.stack(cal)
    jbad = int(np.where(THR == 2400)[0][0])          # 2400 > the big lesion's size
    bound_bad = (RC.shape[0] / (RC.shape[0] + 1.0)) * RC[:, jbad].mean() + 1.0 / (RC.shape[0] + 1.0)
    assert bound_bad > 0.10, f"a 2400-voxel filter must fail the bound (got {bound_bad:.2f})"
    # 4. damage is charged to the filter, not to the model: a perfectly missed satellite
    #    must not appear as filter damage
    gt_only = (big | small | tiny).astype(np.uint8)
    missed = np.zeros_like(gt_only); missed[big] = 2
    r = removal_curve(missed, gt_only, np.array([50.0]), region="WT")
    assert r[0] == 0.0, "voxels the model never predicted must not count as filter damage"
    # 5. the engineering floor caps a statistically valid but unnerving threshold
    d_cap = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR,
                                       component_size_floor=200.0)
    assert d_cap["capped"] and d_cap["threshold"] == 200.0 and d_cap["certified_threshold"] == 1600.0, d_cap
    assert d_cap["threshold"] < big_size
    _c = dict(lam=0.42, n_cal=37, curves=dict(fn=[0.3], fp=[0.1]),
              strata=dict(cut=11.4, n_low=12, n_high=25, lam_low=0.34, lam_high=0.46,
                          feasible_low=True, feasible_high=True,
                          curves_low=dict(fn=[0.2], fp=[0.08]),
                          curves_high=dict(fn=[0.1], fp=[0.05])))
    assert choose_certificate(_c, cnr=9.0)["lam"] == 0.34
    assert choose_certificate(_c, cnr=9.0)["scope"] == "low"
    assert choose_certificate(_c, cnr=15.0)["lam"] == 0.46
    assert choose_certificate(_c, cnr=15.0)["n_cal"] == 25
    assert choose_certificate(_c, cnr=np.nan)["scope"] == "pooled"
    assert choose_certificate(_c, cnr=9.0, stratified=False)["lam"] == 0.42
    _c["strata"]["feasible_low"] = False           # never hand back a dead stratum
    assert choose_certificate(_c, cnr=9.0)["scope"] == "pooled"
    assert choose_certificate(dict(lam=0.5, n_cal=10), cnr=9.0)["scope"] == "pooled"
    print("[ok] choose_certificate: stratum when certifiable, pooled otherwise")

    print(f"       floor: certified {int(d_cap['certified_threshold'])} capped to "
          f"{int(d_cap['threshold'])} by the observed lesion-size floor")
    print(f"  [ok] certified filter: delta={int(d['threshold'])} voxels (< big lesion "
          f"{big_size}), cal bound {d['bound_cal']:.3f}, realised collateral {realised:.3f} "
          f"<= 0.10; a naive delta=2400 is rejected (bound {bound_bad:.2f})")


def _test_metrics():
    big, small, tiny = _toy_case()
    gt = np.zeros(big.shape, np.uint8)
    gt[big] = ED
    gt[small] = ET
    gt[tiny] = ET
    pred = np.zeros_like(gt)
    pred[big] = ED
    pred[small] = ET                      # tiny is missed entirely
    # lesion-wise dice: ET has 2 GT components, one predicted, one missed -> mean in (0,1)
    m, per = lesionwise_dice(pred, gt, region="ET")
    assert len(per) == 2 and 0.0 <= m <= 1.0 and per.min() == 0.0, (m, per)
    st = component_stats(pred, gt, region="ET")
    assert st["missed_lesions"] == 1 and st["false_components"] == 0, st
    # a spurious island must show up as a false component (that is the whole point of the
    # fragmentation metric), while a missed lesion must be counted separately
    pred2 = pred.copy()
    pred2[2:5, 2:5, 2:5] = ET             # spurious blob in the corner, no GT there
    st2 = component_stats(pred2, gt, region="ET")
    assert st2["false_components"] == 1 and st2["missed_lesions"] == 1, st2
    assert st2["n_pred"] == 2 and st2["n_gt"] == 2 and st2["beta0_error"] == 0, st2
    # WT: three disconnected GT blobs, two of them predicted
    st3 = component_stats(pred, gt, region="WT")
    assert st3["n_gt"] == 3 and st3["n_pred"] == 2 and st3["missed_lesions"] == 1, st3
    assert st3["beta0_error"] == 1, st3
    d = dice_regions(pred, gt)
    assert d["ET"] < 1.0 and d["WT"] > 0.5, d
    # HD95 is NaN when one side is empty, and finite otherwise
    assert not np.isnan(hd95(pred, pred, "WT")), "identical masks must give HD95 = 0"
    empty = np.zeros_like(pred)
    assert np.isnan(hd95(empty, gt, "ET")), "undefined HD95 must be NaN, not 0"
    ve = volume_error(pred, gt, "WT")
    assert ve < 0, "this prediction misses a lesion, so the volume error must be negative"
    ba = bland_altman([ve, ve])
    assert ba["n"] == 2 and ba["sd"] == 0.0 and ba["bias"] == ve, ba
    print("  [ok] metrics: lesion-wise Dice, component stats, Dice, HD95 conventions, Bland-Altman")


def _selftest():
    print("safepp self-test")
    _test_postproc_operators()
    _test_metrics()
    _test_envelope_axis()
    _test_crc_guarantee()
    _test_volume_interval()
    _test_stratified()
    _test_certified_filter()
    print("all self-tests passed")


# (self-tests disabled inside the notebook; run code/safepp.py standalone)

# ============================================================================
# F. v2 additions: the structure-recall target and its patch sampler
# ============================================================================
class SkelPatchDataset(PatchDataset):
    """PatchDataset + the thin ground-truth target of the three regions.

    N0 precomputes {pid}_skel.npz (WT/TC/ET thin targets, see skeletonize_mask and
    component_recall_loss) once; training then never has to thin anything on the GPU.  Sample
    i is chosen EXACTLY as in PatchDataset (same seed, same foreground centre, same flips), so
    the target patch is voxel-aligned with the label patch -- `selftest_skel_dataset()`
    verifies that alignment (kept deliberately strict).
    """

    def __init__(self, cache, ids, length, skel_root=None, **kw):
        super().__init__(cache, ids, "base", length, **kw)   # arm is unused here
        self.skel_root = as_roots(skel_root) or self.cache

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        # np.array (a copy), not asarray: with a float32 cache the mmap slice is read-only
        # and the jitter write below would raise; the float16 cache up-casts and copies anyway.
        x = np.array(mri[(slice(None),) + sl], dtype=np.float32)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]
        f = _first_file(self.skel_root, f"{pid}_skel.npz")
        with np.load(f) as z:
            sk = np.stack([np.asarray(z[k][sl], np.float32) for k in ("wt", "tc", "et")])

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad); y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:]); sk = np.pad(sk, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd, sk = (np.flip(a, ax) for a in (x, y, dd, sk))
        # (one draw per axis -- three draws, exactly as PatchDataset does -- so the RNG stream,
        #  and therefore the sample, is identical to the parent's.  The `break` that used to sit
        #  here ended the loop after the first flip, drew a different number of random numbers
        #  than the parent and desynchronised the two samplers from the first flipping sample;
        #  see N2_REVIEW.md, finding F1.  The jitter loop is guarded by self.jitter as the
        #  parent's is, so a jitter=False run consumes no draws either.)
        if self.jitter:
            for c in range(min(4, x.shape[0])):
                if rng.random() < 0.5:
                    m = x[c] != 0
                    x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)),
                torch.from_numpy(np.ascontiguousarray(sk)))


def component_recall_loss(prob, target, eps=1e-6):
    """Soft recall of the PREDICTION on a thin GROUND-TRUTH target (the lesion's core).

        L = 1 - sum(p * T) / sum(T)

    `target` is `centre_ridge(mask)`: the Euclidean-distance ridge of the mask, computed per
    26-connected component (see N0 / `build_skel_*`).  What that buys, in one sentence per
    case:

      * elongated structure (enhancing rim, vessel-like ET, a thin edema arm) -> the ridge is
        a CURVE along its middle, so the term demands that the prediction cover the whole
        structure, not just its fat end;
      * a compact lesion -> the ridge is a short core near its centre;
      * every ground-truth COMPONENT gets at least one ridge voxel, so the term directly
        penalises MISSING A SATELLITE LESION -- a beta_0 (component-count) recall, which is
        the failure mode this project actually has.

    HONEST SCOPE: this is NOT the published Skeleton Recall Loss (ECCV 2024, arXiv
    2404.03010) and must not be quoted as it.  The published method thins the GT with Lee's
    algorithm into a homotopy-equivalent skeleton; the ridge used here is thinner-and-centred
    but not homotopy-preserving.  The FORM is identical (one overlap term, no differentiable
    skeletonisation, a few tensor ops), and `SKEL_BACKEND='skimage'` swaps the target for
    Lee's thinning when that build passes `_skimage_skeleton_ok()` -- i.e. you can run the
    published variant from the same code, and N0 prints which target it built.

    `prob` and `target` are (B, C, D, H, W); the reduction is over voxels and classes.
    """
    # the sums ask for fp32 explicitly: `torch.sum` on a half tensor returns a half, and a
    # whole-patch sum of fp16 values overflows 65504 -> inf -> nan (exactly the Dice problem)
    num = (prob * target).sum(dtype=torch.float32)
    den = target.sum(dtype=torch.float32) + eps
    return 1.0 - num / den


# old name kept so nothing that already imports it breaks
skeleton_recall_loss = component_recall_loss


class DiceLossFast(nn.Module):
    """MONAI `DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)`
    reimplemented so the ELEMENTWISE maths stays in the autocast dtype (fp16 on GPU = fast) while
    every REDUCTION accumulates in fp32.  Verified against MONAI in fp32 (same value to ~1e-5,
    `code/test_dice_fast.py`).

    Why this exists -- it is not a micro-optimisation, it is the bug that killed two GPU runs:
    the reduction in MONAI's DiceLoss is `torch.sum(half_tensor)`, which ACCUMULATES in fp32 but
    RETURNS A HALF.  Over a 96^3 patch that result is ~1.8e5, i.e. above fp16's 65504 -> inf ->
    nan loss -> nan weights -> a model that predicts nothing for the whole run (`loss nan` for
    8000 iterations, `best=0.0556`, in both arms).  Casting the whole loss to fp32 fixes it but
    makes the elementwise ops (and their backward) fp32 as well, measured ~5x slower per
    iteration on the user's T4 (0.33 -> 1.6 s/it).  Here the sums ask for fp32 explicitly:
    `(p * p).sum(dtype=torch.float32)`, so nothing overflows and nothing is wasted.
    """

    def __init__(self, smooth_nr=0.0, smooth_dr=1e-5, batch=True):
        super().__init__()
        self.smooth_nr, self.smooth_dr, self.batch = float(smooth_nr), float(smooth_dr), bool(batch)

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)                       # autocast dtype: fp16 on GPU
        y = targets if targets.dtype == p.dtype else targets.to(p.dtype)
        dims = list(range(2, p.ndim)) + ([0] if self.batch else [])
        # MONAI's order and formula, so the value matches to rounding
        intersection = (p * y).sum(dim=dims, dtype=torch.float32)
        ground_o = (y * y).sum(dim=dims, dtype=torch.float32)
        pred_o = (p * p).sum(dim=dims, dtype=torch.float32)
        f = 1.0 - (2.0 * intersection + self.smooth_nr) / (ground_o + pred_o + self.smooth_dr)
        return f.mean()


class TopologyLoss(nn.Module):
    """dice+BCE plus an optional structure-recall term whose strength is SCHEDULED by the
    training loop (`self.w` is a plain float, ramped from 0 to the arm's nominal value).

    Why not a learnable multiplier -- measured, not assumed: with `w` as an nn.Parameter the
    gradient of the loss w.r.t. w is +R(theta) > 0 at whatever theta the model currently has,
    so plain gradient descent drives w below zero, relu(w) kills the term and the arm becomes
    silently identical to its control (observed: w = -0.006 after 25 iterations, i.e. off).
    A learnable weight can only earn its place through a meta-gradient over the whole training
    run, which a single scalar under SGD does not have.

    What is kept is the property that actually matters: w starts at ZERO, so step 0 and the
    whole warm-up phase are bit-identical to the Dice+BCE control, and whether the term earns
    its place is decided by the held-out metrics (N2/N3), not by the scalar itself.
    """

    def __init__(self, w0=0.0, boundary=True):
        super().__init__()
        self.dice = DiceLossFast()      # fp16-safe + fast; same value as MONAI's DiceLoss
        self.boundary = boundary
        self.w = float(w0)                       # ramped by the training loop; not a parameter
        self.nominal = float(w0)

    def forward(self, logits, y, dist=None, skel=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist)
                       if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if skel is not None and float(self.w) > 0:
            loss = loss + float(self.w) * component_recall_loss(torch.sigmoid(logits), skel)
        return loss


def _skimage_skeleton_ok():
    """Is this build's skimage.skeletonize trustworthy?

    NOT a formality: builds exist that return an ALL-ZERO skeleton for shapes as simple as a
    solid cube (observed in the wild with scikit-image 0.25.x).  A silently empty skeleton
    would make the topology arm train on nothing, so we detect it and fall back.
    Returns (ok, note).
    """
    try:
        from skimage.morphology import skeletonize
    except Exception as e:                                   # pragma: no cover
        return False, f"skimage unavailable ({e})"
    zz, yy, xx = np.mgrid[0:32, 0:32, 0:32]
    probes = (("cube8", np.ones((8, 8, 8), bool), 4),
              ("tube4x4x17", _probe_tube(), 8),
              ("ellipsoid5x6x7", (((zz - 16) / 5.0) ** 2 + ((yy - 16) / 6.0) ** 2
                                  + ((xx - 16) / 7.0) ** 2) <= 1, 4))
    notes = []
    for name, m, floor in probes:
        try:
            n = int(np.asarray(skeletonize(m.astype(bool))).sum())
        except Exception as e:                               # pragma: no cover
            return False, f"skeletonize raised on {name} ({e})"
        if n < floor:
            return False, (f"skeletonize is degenerate here: {name} ({int(m.sum())} voxels "
                           f"in) -> {n} skeleton voxel(s), expected >= {floor}")
        notes.append(f"{name}={n}")
    return True, "ok (" + ", ".join(notes) + ")"


def _probe_tube():
    t = np.zeros((10, 10, 20), bool)
    t[3:7, 3:7, 2:19] = True
    return t


def centre_ridge(mask):
    """Dependency-free centre line: the ridge of the Euclidean distance transform.

    Voxels that sit inside the mask and are a local maximum of the distance to the background
    in their 26-neighbourhood.  For a tube that is a curve, for a blob a medial sheet -- both
    are thin and centred, which is what the skeleton-recall term needs (it asks how much of
    the centre of the lesion the prediction covers).

    Guarantee: the result is a subset of the mask and is never empty when the mask is not.
    It is NOT a homotopy-equivalent thinning; `skeletonize_mask` prefers Lee thinning when
    that is available and trustworthy.
    """
    from scipy import ndimage
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    # pad by one background voxel: without it the distance transform of a mask that touches
    # the array border measures the distance to the array corner instead of to the boundary
    mp = np.pad(m, 1)
    d = ndimage.distance_transform_edt(mp)
    ridge = mp & (d >= ndimage.maximum_filter(d, size=3))     # local maxima of the distance
    if not ridge.any():                                       # single-voxel mask, etc.
        ridge = mp & (d == d.max())
    lab, n = ndimage.label(mp, structure=np.ones((3, 3, 3), int))
    for i in range(1, n + 1):                                 # every component represented
        comp = lab == i
        if not (ridge & comp).any():
            ridge |= comp & (d == d[comp].max())
    return ridge[1:-1, 1:-1, 1:-1]


def skeletonize_mask(mask, backend="auto", verbose=False):
    """Thin, centred skeleton of `mask`, with a self-checked backend.

    backend='auto'             skimage's 3-D Lee thinning if that build passes a sanity check,
                               otherwise the dependency-free distance-transform ridge
    backend='skimage'|'ridge'  force one
    """
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    if backend == "auto":
        ok, note = _skimage_skeleton_ok()
        backend = "skimage" if ok else "ridge"
        if verbose:
            print(f"  skeleton backend: {backend} ({note})")
    if backend == "skimage":
        from skimage.morphology import skeletonize
        out = np.asarray(skeletonize(m.astype(bool))).astype(bool)
        if out.any():
            return out
    return centre_ridge(m)


def skeleton_selftest(verbose=True):
    """Fail loudly if the skeletons the topology arm will train on are degenerate."""
    zz, yy, xx = np.mgrid[0:24, 0:24, 0:24]
    blob = (((zz - 11) / 5.0) ** 2 + ((yy - 11) / 6.0) ** 2 + ((xx - 11) / 7.0) ** 2) <= 1
    tube = np.zeros((24, 24, 24), bool)
    tube[10:14, 10:14, 3:21] = True
    out = {}
    for name, m in (("blob", blob), ("tube", tube), ("blob+tube", blob | tube),
                    ("cube8", np.ones((8, 8, 8), bool))):
        sk = skeletonize_mask(m)
        assert sk.any(), f"{name}: empty skeleton for a non-empty mask"
        assert not (sk & ~m).any(), f"{name}: skeleton outside the mask"
        frac = sk.sum() / max(1, int(m.sum()))
        assert frac < 0.9, f"{name}: skeleton is not thinner than the mask ({frac:.2f})"
        out[name] = (int(m.sum()), int(sk.sum()))
    ok, note = _skimage_skeleton_ok()
    if verbose:
        print(f"[ok] skeletonize_mask {out} | skimage backend usable: {ok} ({note})")
    return out


def selftest_skel_dataset(cache, ids, patch=(32, 32, 32), n=8, seed=42, skel_root=None):
    """Verify the skeleton patch is voxel-aligned with the targets (skel[0] must be the WT
    skeleton, i.e. nonzero exactly where y[0] is nonzero for a non-empty target patch)."""
    a = PatchDataset(cache, ids, "base", length=n, patch=patch, seed=seed, dod_tag=None)
    b = SkelPatchDataset(cache, ids, length=n, patch=patch, seed=seed, skel_root=skel_root)
    for i in range(n):
        x1, y1, d1 = a[i]
        x2, y2, d2, s = b[i]
        assert torch.equal(y1, y2), f"targets differ at sample {i}"
        assert torch.equal(d1, d2), f"distance maps differ at sample {i}"
        assert s.shape == y1.shape, (s.shape, y1.shape)
        # WT skeleton must sit inside the WT target (allowing the 1-voxel skeleton margin)
        assert float((s[0] * (1 - y1[0])).sum()) <= 0.02 * float(y1[0].sum()) + 5, \
            f"skeleton not aligned with the target at sample {i}"
    return True


# ---- aliases: the cells below say bratsx.x() / safepp.x() -------------------
# In a notebook this cell runs in __main__, so both names are simply THIS
# namespace: nothing to import, nothing to attach, and a forked DataLoader
# worker resolves them for free.  (The fallback covers the case where the
# library is exec'd somewhere other than __main__, e.g. a test harness.)
import sys as _sys, types as _types
_ns = globals()
bratsx = _sys.modules.get(__name__)
if getattr(bratsx, "__dict__", None) is not _ns:
    bratsx = _types.ModuleType("bratsx")
    bratsx.__dict__.update(_ns)
safepp = bratsx
print("library ready:", len([k for k in dir(bratsx) if not k.startswith('_')]),
      "names (bratsx and safepp are aliases of this notebook's namespace)")


library ready: 167 names (bratsx and safepp are aliases of this notebook's namespace)


In [19]:
# %% [CELL 4] LOCATE RUNS / CACHE, LOAD THE SPLIT -----------------------------
# CORRECTED BUILD: N2 2026-10-09a + G1 (separate best.pt selected step from completed N1 budget)
import glob, json, os, shutil, time
import numpy as np
import pandas as pd
import torch

os.makedirs(WORK, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def _find_all(tag, probe):
    hits = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
            if os.path.isdir(p) and os.path.basename(p).startswith(tag)
            and glob.glob(os.path.join(p, probe))]
    return sorted(set(hits))


# Attach exactly ONE runs dataset.  The search is recursive and takes the first hit in sorted path
# order, so a stale runs output left attached is used SILENTLY -- so say which one won.
_RUNS_HITS = _find_all("runs", os.path.join("*", "best.pt"))
if len(_RUNS_HITS) > 1:
    print(f"!! {len(_RUNS_HITS)} run sources under {INPUT_ROOT}:")
    for _q in _RUNS_HITS:
        print(f"     {_q}")
    print("   -> using the first.  The fingerprint below must show the iteration count your last "
          "N1 session ended at, and the SAME count for both arms; otherwise detach the stale runs "
          "dataset and re-run this cell.")
RUNS = RUNS_INPUT or next(iter(_RUNS_HITS), None) or os.path.join(WORK, "runs")
CACHE_BASE = bratsx.as_roots(CACHE_BASE) or _find_all("cache_base", "*_mri.npy") \
    or bratsx.as_roots(os.path.join(WORK, "cache_base"))
assert CACHE_BASE, f"cache not found under {INPUT_ROOT}: attach the N0 output dataset(s)"
print(f"runs:  {RUNS}\ncache: {len(CACHE_BASE)} shard(s)")

CKPTS = {}
for ck in sorted(glob.glob(os.path.join(RUNS, "*", "best.pt"))):
    arm, _, seed = os.path.basename(os.path.dirname(ck)).rpartition("_seed")
    CKPTS.setdefault(arm, {})[str(seed)] = ck
for a in sorted(CKPTS):
    print(f"  {a:8s} {len(CKPTS[a])} seed(s): {sorted(CKPTS[a])}")
# NOTE: checkpoints are NOT required here.  They are needed only to COMPUTE probabilities that
# are missing; if a `probs` dataset (Add Data) already covers every case, cell 5 adopts it and no
# model is ever built.  So this is a status line, and the hard check happens after the split is
# known (there we can say exactly which arm is short).
if not CKPTS:
    print("no training runs attached -> this session must certify from a `probs` dataset")
else:
    print("arms with checkpoints:", ", ".join(sorted(CKPTS)))


def _ckpt_fingerprint(path):
    """Identity of a checkpoint: (iter, best, sha1 over the model tensors in key order).

    The hash is over the WEIGHTS, not the file: the same run re-saved or re-downloaded hashes the
    same, and any change in training -- more iterations, a different seed, a different rollout --
    hashes differently.  That is exactly the question a cached `probs` dataset raises.
    """
    import hashlib
    st = torch.load(path, map_location="cpu", weights_only=False)
    m = st.get("model", st)
    if not isinstance(m, dict) or not m:
        return None
    h = hashlib.sha1()
    for k in sorted(m):
        v = m[k]
        try:
            arr = v.detach().to("cpu").contiguous().numpy()
        except Exception:
            arr = np.asarray(v)
        h.update(k.encode())
        h.update(np.ascontiguousarray(arr).tobytes())
    try:
        best_f = round(float(st.get("best", float("nan"))), 4)
    except Exception:
        best_f = float("nan")
    return dict(it=int(st.get("iter", -1)), best=best_f, sha1=h.hexdigest()[:12])


def _walk_run_records(obj):
    """Yield JSON dictionaries recursively (DONE files can wrap the run record)."""
    if isinstance(obj, dict):
        if "iters" in obj:
            yield obj
        for _v in obj.values():
            yield from _walk_run_records(_v)
    elif isinstance(obj, list):
        for _v in obj:
            yield from _walk_run_records(_v)


def _n1_training_completion(best_path, arm, seed):
    """Final N1 update count, distinct from the validation-selected step in best.pt.

    Prefer N1's DONE/run JSON (`iters` is the completed total); fall back to last.pt.  The
    `best.pt` `iter` is only the step at which that arm's best validation snapshot was saved,
    and can legitimately differ across arms even when they both completed the same budget.
    """
    _run_dir = os.path.dirname(best_path)
    _fixed = ("DONE", "DONE.json", "done.json", "run.json", "run_summary.json",
              "summary.json", "status.json")
    _cands = [os.path.join(_run_dir, _n) for _n in _fixed]
    _cands += sorted(_p for _p in glob.glob(os.path.join(_run_dir, "*.json"))
                     if os.path.basename(_p).lower() not in {"history.json"})
    _cands += sorted(glob.glob(os.path.join(_run_dir, "DONE*")))
    _seen = set()
    for _p in _cands:
        if _p in _seen or not os.path.isfile(_p):
            continue
        _seen.add(_p)
        try:
            with open(_p, "r", encoding="utf-8") as _f:
                _obj = json.load(_f)
        except Exception:
            continue
        for _rec in _walk_run_records(_obj):
            _ra = _rec.get("arm")
            _rs = _rec.get("seed")
            if _ra is not None and str(_ra) != str(arm):
                continue
            if _rs is not None and str(_rs) != str(seed):
                continue
            try:
                _it = int(_rec["iters"])
            except (TypeError, ValueError, KeyError):
                continue
            if _it >= 0:
                return {"iters": _it, "source": os.path.basename(_p)}

    _last = os.path.join(_run_dir, "last.pt")
    if os.path.isfile(_last):
        try:
            _st = torch.load(_last, map_location="cpu", weights_only=False)
            if isinstance(_st, dict):
                _it = _st.get("iter", _st.get("iters"))
                if _it is not None and int(_it) >= 0:
                    return {"iters": int(_it), "source": "last.pt"}
        except Exception as _e:
            print(f"  {arm}/seed{seed}: could not read last.pt ({_e})")
    return None


CKPT_FP = {}
for _a in sorted(CKPTS):
    # N2 uses the first seed of each arm in the report.  Keep its selected best.pt identity.
    _ss = sorted(CKPTS[_a])
    if not _ss:
        continue
    _s = _ss[0]
    try:
        _fp = _ckpt_fingerprint(CKPTS[_a][_s])
        if _fp:
            CKPT_FP[_a] = _fp
    except Exception as _e:                      # a truncated or foreign file must not stop N2
        print(f"  {_a}/seed{_s}: could not fingerprint best.pt ({_e})")

TRAINING_FP = {}
for _a in sorted(CKPTS):
    _ss = sorted(CKPTS[_a])
    if not _ss:
        continue
    _s = _ss[0]
    _meta = _n1_training_completion(CKPTS[_a][_s], _a, _s)
    if _meta:
        TRAINING_FP[_a] = {"seed": _s, **_meta}
    else:
        print(f"  !! {_a}/seed{_s}: no N1 completion count found (checked DONE/run JSON and last.pt)")

if CKPT_FP:
    print("best checkpoint fingerprint (validation-selected step): " + " | ".join(
        f"{a} selected_it={d['it']} best={d['best']} sha1={d['sha1']}"
        for a, d in sorted(CKPT_FP.items())))
    print("   note: best.pt selected_it may differ across arms; it is NOT the completed N1 budget.")
if TRAINING_FP:
    print("training completion: " + " | ".join(
        f"{a}/seed{d['seed']} iters={d['iters']} [{d['source']}]"
        for a, d in sorted(TRAINING_FP.items())))
    _done_iters = {a: d["iters"] for a, d in sorted(TRAINING_FP.items())}
    if len(set(_done_iters.values())) > 1:
        print(f"!! the arms completed different N1 budgets ({_done_iters}); finish the short arm "
              f"before certifying.")
elif CKPT_FP:
    print("!! N1 completion budget could not be verified from this runs output; CHECK 2/3 will "
          "stop before inference. Attach the complete N1 output containing DONE metadata or last.pt.")

_sp = next((q for q in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "dataset_split.json"),
                                        recursive=True)) + [os.path.join(WORK, "dataset_split.json")]
            if os.path.exists(q)), None)
assert _sp, "dataset_split.json not found: attach the cache dataset"
# The glob above is in SORTED PATH ORDER: when two attached datasets both carry a split file, the
# one whose path sorts first wins and the other is ignored without a word.  Print the winner.
print(f"split: {_sp}")
split = json.load(open(_sp))
VAL_IDS = [p for p in split["val"] if bratsx.has_case(CACHE_BASE, p)]
TEST_IDS = [p for p in split["test"] if bratsx.has_case(CACHE_BASE, p)]
_mfs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "manifest_base*.csv"), recursive=True))
if not _mfs:
    _mfs = sorted(glob.glob(os.path.join(WORK, "manifest_base*.csv")))
man = (pd.concat([pd.read_csv(f) for f in _mfs]).drop_duplicates("pid").set_index("pid")
       if _mfs else pd.DataFrame())
print(f"val {len(VAL_IDS)} | test {len(TEST_IDS)} | manifest {len(man)} rows")
assert VAL_IDS, "no validation cases: the certificates need a calibration set"
if not len(man):
    print("!! manifest missing -> the contrast stratification will be skipped")

LAMBDAS = np.round(LAMBDA_GRID if LAMBDA_GRID is not None else np.linspace(0.05, 0.95, 19), 3)
# DELETED ON PURPOSE -- do not re-add.  This cell used to build
#     GT     = {pid: seg > 0 ...}                      for every val + test case
#     GT_REG = {region: {pid: np.isin(seg, REGIONS[region]) ...}}
# Nothing downstream read either name: risk_curves, evaluate, tune_post and the sanity check all
# re-load `..._seg.npy` where they need it.  As bool arrays over 74 cases that is ~74 x 4 volumes
# ~ 2.6 GB held for the whole session, and the failure mode is an out-of-memory kill DURING
# calibration -- after the expensive inference has already run.
print(f"lambda grid: {LAMBDAS[0]}..{LAMBDAS[-1]} ({len(LAMBDAS)} values)")


device: cuda
runs:  /kaggle/input/datasets/thinkbin/brats-run-v2/runs
cache: 1 shard(s)
  base     1 seed(s): ['42']
  sr50     1 seed(s): ['42']
arms with checkpoints: base, sr50
best checkpoint fingerprint (validation-selected step): base selected_it=19500 best=0.8191 sha1=418a2c28a16c | sr50 selected_it=12500 best=0.8167 sha1=24298f8183bc
   note: best.pt selected_it may differ across arms; it is NOT the completed N1 budget.
training completion: base/seed42 iters=21795 [DONE] | sr50/seed42 iters=21795 [DONE]
split: /kaggle/input/datasets/thinkbin/brats-cache-v2/dataset_split.json
val 37 | test 37 | manifest 369 rows
lambda grid: 0.05..0.95 (19 values)


In [20]:
# %% [CELL 5] INFERENCE -- probabilities cached once for val and test ---------
# The probabilities are the expensive part (a full sliding-window pass over val+test).  If a
# previous N2 output is attached (Add Data -> Notebook Output Files, or a dataset made from that
# output), adopt it and copy it into /kaggle/working/probs so this cell has nothing to compute:
# a re-certification then costs minutes, not an hour, and needs no GPU training.
PROBS = os.path.join(WORK, "probs")


def _prob_roots():
    """Every directory that has the SHAPE of a probability tree: it holds `<arm>_seed<k>/*.npz`.

    A Kaggle dataset built from an N1/N2 output can land anywhere under /kaggle/input -- its own
    root may hold `probs/`, or the arm folders directly, or the output may be re-nested a level.
    Guessing a path is how a correct dataset ends up "not found", so we look for the shape instead.
    Read-only dataset roots are never written to: everything is copied into /kaggle/working/probs,
    which also keeps room for the derived ensemble below.
    """
    roots = []
    for p in sorted(glob.glob(os.path.join(INPUT_ROOT, os.sep.join(["**"])), recursive=True)):
        if not os.path.isdir(p) or os.path.abspath(p) == os.path.abspath(PROBS):
            continue
        if glob.glob(os.path.join(p, "*_seed*", "*.npz")):
            roots.append(p)                                    # arm folders sit in p
        elif glob.glob(os.path.join(p, "*", "*_seed*", "*.npz")):
            roots.append(p)                                    # p/probs/<arm>_seed<k>
    return roots


def _adopt_probs():
    """Copy attached probability maps (and their recipe stamp) into /kaggle/working/probs.

    `_provenance.json` MUST come along: without it the adopted cache looks like it was made by a
    different inference recipe, gets moved aside as `__stale`, and the notebook recomputes the very
    thing the attached dataset existed to avoid.
    """
    got = 0
    for src in _prob_roots():
        for d in sorted(glob.glob(os.path.join(src, "*"))):
            if not os.path.isdir(d) or "__stale" in d:
                continue                        # a quarantined cache is not a source
            if not glob.glob(os.path.join(d, "*.npz")):
                continue                        # `src` may itself be a wrapper folder: without this
                                                # the whole tree is copied a second time under
                                                # working/probs/<name>/ (a multi-GB dataset twice)
            dst = os.path.join(PROBS, os.path.basename(d))
            os.makedirs(dst, exist_ok=True)
            for f in glob.glob(os.path.join(d, "*.npz")) + glob.glob(os.path.join(d, "_provenance.json")):
                t = os.path.join(dst, os.path.basename(f))
                if not os.path.exists(t):
                    try:
                        shutil.copy2(f, t)
                        got += 1
                    except Exception as e:                     # a read-only mount, a bad file, ...
                        print(f"   ! could not copy {f}: {e}")
        print(f"[probs] adopted from {src}")
    return got


_n_adopted = 0 if glob.glob(os.path.join(PROBS, "*", "*.npz")) else _adopt_probs()
if _n_adopted:
    print(f"[probs] {_n_adopted} file(s) copied into {PROBS} -- nothing to compute means no "
          f"checkpoint, no GPU, no sliding window")
elif not glob.glob(os.path.join(PROBS, "*", "*.npz")):
    print("[probs] no cached probability maps found: attach a `probs` dataset (Add Data -> the "
          "dataset made from an earlier N2 output) or the N1 runs, or this session has to run "
          "inference and needs the checkpoints.")


_RECIPE_KEYS = ("flips", "overlap", "tta_impl")


def _provenance():
    return dict(flips=[list(f) for f in (TTA_FLIPS or [()])], overlap=float(INFER_OVERLAP),
                tta_impl=int(getattr(bratsx, "TTA_IMPL", 0)))


def _stamp(arms=None):
    """What gets written next to a cache: the inference recipe AND the identity of the weights it
    was made from.  The recipe alone cannot tell two trainings apart -- an older `best.pt` produces
    the same recipe and a different model, and the certificate would then describe weights that are
    not the ones attached to the session."""
    st = {"recipe": _provenance()}
    arms = sorted(CKPT_FP) if arms is None else [a for a in arms if a in CKPT_FP]
    if arms:
        st["ckpt"] = {a: CKPT_FP[a] for a in arms}
    return st


def _recipe_of(obj):
    """Read the recipe out of a stamp, old or new.  Old stamps (published by an earlier build) are
    flat `{"flips": ..., "overlap": ..., "tta_impl": ...}`: they must still verify, or every cache
    made before this fix would be quarantined and recomputed for no reason."""
    if isinstance(obj, dict) and isinstance(obj.get("recipe"), dict):
        obj = obj["recipe"]
    if not isinstance(obj, dict):
        return None
    return {k: obj.get(k) for k in _RECIPE_KEYS}


def _provenance_ok(out):
    """A cache is reusable only if it was produced by the SAME inference recipe.  Reusing a
    plain-inference cache after switching flip averaging on would mix two definitions of the
    arm's probabilities, and nothing downstream could tell."""
    f = os.path.join(out, "_provenance.json")
    if not os.path.exists(f):
        return False
    try:
        return _recipe_of(json.load(open(f))) == _provenance()
    except Exception:
        return False


def _fp_txt(d):
    if not isinstance(d, dict):
        return "no fingerprint"
    return f"it={d.get('it')} sha1={d.get('sha1')}"


def _ckpt_state(out, arm):
    """Was this cache made from the checkpoint attached to THIS session?

    'match'     the stamp carries a fingerprint and it equals the attached checkpoint
    'mismatch'  the stamp carries a fingerprint and it differs -> these maps are the old model's
    'unknown'   the stamp predates fingerprinting (or this arm is not part of it) -> cannot tell
    'no-ckpt'   nothing attached to compare against; the probs-only route cannot verify identity
    """
    stamped = None
    f = os.path.join(out, "_provenance.json")
    if os.path.exists(f):
        try:
            obj = json.load(open(f))
            stamped = obj.get("ckpt") if isinstance(obj, dict) else None
        except Exception:
            stamped = None
    if not CKPT_FP:
        return "no-ckpt", stamped, None
    if not stamped or arm not in stamped or arm not in CKPT_FP:
        return "unknown", stamped, None
    if stamped[arm] == CKPT_FP[arm]:
        return "match", stamped, CKPT_FP[arm]
    return "mismatch", stamped, CKPT_FP[arm]


# The probe compares one cached map with the same case recomputed from the attached checkpoint.
# The MEAN absolute difference is the wrong statistic: most of the volume is background where both
# models output ~0, so two genuinely different networks averaged only 2.7e-3 apart in testing.  The
# 99.9th percentile of |dp| is where the disagreement lives (tumour, border, false positives) and it
# is ~0 for identical weights, including across devices.
PROBE_TOL = 2e-3


def _quarantine(out, why):
    """Keep the maps but get them out of the way.  An older model's probabilities are evidence, not
    rubbish -- and leaving them in place would make the recompute below a no-op that silently
    recertifies the old model, which is the exact bug this block exists to prevent."""
    stale = out + "__stale"
    if os.path.isdir(stale):
        shutil.rmtree(stale)
    shutil.move(out, stale)
    os.makedirs(out, exist_ok=True)
    print(f"{why} -> moved to {os.path.basename(stale)} and recomputing from the attached runs")


def _probe_identity(arm, seed, out, ids):
    """A cache stamped by an older build carries no fingerprint, so recipe verification passes and
    the question "is this the model I have attached?" is left open -- which is the one way this
    route can certify the wrong weights without noticing.

    Answer it directly: recompute ONE case from the attached checkpoint and compare it with the
    cached map.  Seconds instead of the full set, and the answer is yes/no instead of a shrug.
    Returns (case_id, mean_abs_diff) or None if the probe could not be run.
    """
    pid = next((p for p in ids if os.path.exists(os.path.join(out, f"{p}.npz"))), None)
    if pid is None or arm not in CKPT_FP:
        return None
    try:
        model = bratsx._segresnet().to(device)
        st = torch.load(CKPTS[arm][seed], map_location=device, weights_only=False)
        model.load_state_dict(st["model"]); model.eval()
        x, _, _ = bratsx.loader_case(CACHE_BASE, pid)
        pr = bratsx.infer_probs_tta(model, x, PATCH, INFER_OVERLAP, device, flips=TTA_FLIPS,
                                    tag=f"{arm}/seed{seed} identity probe {pid}")
        st_ = dict(mean=0.0, p999=0.0, mx=0.0)
        with np.load(os.path.join(out, f"{pid}.npz")) as z:
            for i, k in enumerate(("p_wt", "p_tc", "p_et")):
                diff = np.abs(z[k].astype(np.float32) - np.asarray(pr[i], dtype=np.float32))
                st_["mean"] = max(st_["mean"], float(diff.mean()))
                st_["p999"] = max(st_["p999"], float(np.percentile(diff, 99.9)))
                st_["mx"] = max(st_["mx"], float(diff.max()))
        del model
        return pid, st_
    except Exception as e:                    # a probe must never be the reason a run dies
        print(f"[{arm}/{seed}] identity probe could not run ({e})")
        return None


def _identity_line(state, stamped, arm, mine):
    if state == "match":
        return (f"model identity: matches the attached checkpoint "
                f"({arm} {_fp_txt(mine)})")
    if state == "mismatch":
        return (f"model identity: MADE FROM A DIFFERENT CHECKPOINT "
                f"(cache {_fp_txt((stamped or {}).get(arm))} vs attached {_fp_txt(mine)})")
    if state == "no-ckpt":
        return ("model identity: cannot be checked in this session "
                "(no checkpoints attached -- only the maps are here)")
    return ("model identity: NOT verifiable -- these maps were cached by an older build that did "
            "not record which weights made them.  If N1 has been re-run since this cache was "
            "published, attach the fresh N2 output (or delete the probs dataset and re-run here "
            "with the runs attached); if the runs are unchanged, there is nothing to do.")


def infer_all(arm, seed, ids, tag):
    """Sliding-window probabilities, one .npz per case, with optional flip averaging.

    Cached: re-running this cell is cheap because existing files are skipped -- but only while
    the inference recipe is unchanged.  If the recipe changed (e.g. flip averaging was switched
    on), the old maps are moved aside rather than deleted, so an earlier run stays available
    for the adoption path.
    """
    out = os.path.join(PROBS, f"{arm}_seed{seed}")
    have = [q for q in ids if os.path.exists(os.path.join(out, f"{q}.npz"))]
    todo_now = [p for p in ids if not os.path.exists(os.path.join(out, f"{p}.npz"))]
    if not todo_now:
        # Nothing to compute: this is the "probs as a dataset" route.  Report exactly what could be
        # verified -- the inference recipe, and the identity of the weights -- and act on it:
        #   stamp says different weights   -> recompute (checkpoints are attached)
        #   recipe changed                 -> recompute (checkpoints are attached)
        #   old stamp, no fingerprint      -> PROBE one case, then decide
        #   no checkpoints attached        -> use the maps, say the identity is unverifiable
        state, stamped, mine = _ckpt_state(out, arm)
        recipe_ok = _provenance_ok(out)
        print(f"[{arm}/{seed}] {tag}: {len(ids)} cached map(s), " + (
            f"recipe verified ({_provenance()})" if recipe_ok else
            "no matching _provenance.json (an older build wrote this cache)"))
        recompute = False
        said = False                       # the probe prints its own verdict; do not repeat it
        if CKPT_FP and state == "mismatch":
            print(f"[{arm}/{seed}] {tag}: the cache was made from a DIFFERENT checkpoint "
                  f"(cache {_fp_txt((stamped or {}).get(arm))} vs attached {_fp_txt(mine)})")
            recompute = True
        elif CKPT_FP and not recipe_ok:
            print(f"[{arm}/{seed}] {tag}: the inference recipe changed since this cache was made, "
                  f"and the checkpoints to recompute it are attached")
            recompute = True
        elif CKPT_FP and state == "unknown":
            pr = _probe_identity(arm, seed, out, ids)
            if pr is not None:
                pid, _st = pr
                said = True
                _txt = (f"|dp| mean {_st['mean']:.2e}, p99.9 {_st['p999']:.2e}, "
                        f"max {_st['mx']:.2e}")
                if _st["p999"] <= PROBE_TOL:
                    print(f"[{arm}/{seed}] {tag}: model identity: PROBED on {pid} -- one case "
                          f"recomputed from the attached checkpoint matches the cache "
                          f"({_txt}; p99.9 <= {PROBE_TOL:g}) -> these ARE the attached weights.  "
                          f"Fingerprint written into the stamp, so the probe is a one-off cost.")
                    try:
                        json.dump(_stamp([arm]),
                                  open(os.path.join(out, "_provenance.json"), "w"))
                    except Exception as e:
                        print(f"[{arm}/{seed}] could not upgrade the stamp: {e}")
                else:
                    print(f"[{arm}/{seed}] {tag}: model identity: PROBED on {pid} -- one case "
                          f"recomputed from the attached checkpoint DIFFERS from the cache "
                          f"({_txt}; p99.9 > {PROBE_TOL:g}) -> these are some other weights")
                    recompute = True
        if recompute:
            _quarantine(out, f"[{arm}/{seed}] {tag}: the cached maps are not the attached "
                             f"checkpoint's output")
            have, todo_now = [], list(ids)          # fall through to the inference path below
        else:
            if not said:
                print(f"[{arm}/{seed}] {tag}: " + _identity_line(state, stamped, arm, mine))
            return out
    assert arm in CKPTS and seed in CKPTS[arm], (
        f"[{arm}/{seed}] {len(todo_now)} of {len(ids)} {tag} case(s) have no cached "
        f"probability map and no checkpoint in {RUNS}.  Attach ONE of: (a) the N1 runs output "
        f"(best.pt per arm), or (b) a probs dataset that covers every val case"
        + (" and every test case" if tag == "test" else "")
        + f".  Missing examples: {todo_now[:5]}")
    if have and not _provenance_ok(out):
        stale = out + "__stale"
        if os.path.isdir(stale):
            shutil.rmtree(stale)
        shutil.move(out, stale)
        print(f"[{arm}/{seed}] {len(have)} cached map(s) used a different inference recipe -> "
              f"moved to {os.path.basename(stale)} and recomputed")
    # CORRECTED BUILD (cell-5 insert): partial-cache checkpoint identity -- see N2_REVIEW.md
    # CORRECTED BUILD (cell-5 insert): the identity question the full-cache branch asks above, for
    # the PARTIAL cache.  Same reasoning -- a matching recipe says nothing about WHICH weights wrote
    # the maps already on disk -- and the same consequence if it is skipped: the missing cases are
    # computed from the attached checkpoint and the stamp is rewritten, so the arm's probabilities
    # end up averaged from two models and labelled as one.
    #
    # `state`/`stamped`/`mine` are read BEFORE the recipe branch below on purpose: that branch can
    # call _quarantine(), which moves `out` aside and recreates it empty -- after that,
    # _ckpt_state(out, arm) looks at the brand-new empty folder and answers "unknown" even though
    # the stamp was there a moment ago, and a mismatched partial cache would slip through.
    state, stamped, mine = _ckpt_state(out, arm)
    if have and not _provenance_ok(out):
        stale = out + "__stale"
        if os.path.isdir(stale):
            shutil.rmtree(stale)
        shutil.move(out, stale)
        print(f"[{arm}/{seed}] {len(have)} cached map(s) used a different inference recipe -> "
              f"moved to {os.path.basename(stale)} and recomputed")
    if not have or not CKPT_FP:
        # nothing cached to identify, or nothing to identify it against: the recipe line above
        # already said what could be said, and the assert above covers a missing checkpoint.
        if have and not CKPT_FP:
            print(f"[{arm}/{seed}] {tag}: model identity: cannot be checked in this session "
                  f"(no checkpoints attached -- only the maps are here)")
    elif state == "mismatch":
        _quarantine(out, f"[{arm}/{seed}] {tag}: the {len(have)} cached map(s) were made from a "
                         f"DIFFERENT checkpoint (cache {_fp_txt((stamped or {}).get(arm))} vs "
                         f"attached {_fp_txt(mine)})")
        have, todo_now = [], list(ids)
    elif state == "unknown":
        pr = _probe_identity(arm, seed, out, ids)
        if pr is not None:
            _pid, _st = pr
            if _st["p999"] <= PROBE_TOL:
                print(f"[{arm}/{seed}] {tag}: model identity: PROBED on {_pid} -- the cached map "
                      f"matches the attached checkpoint (p99.9 {_st['p999']:.2e} <= "
                      f"{PROBE_TOL:g}), so keeping it and computing the missing cases here is safe")
            else:
                print(f"[{arm}/{seed}] {tag}: model identity: PROBED on {_pid} -- the cached map "
                      f"DIFFERS from the attached checkpoint (p99.9 {_st['p999']:.2e} > "
                      f"{PROBE_TOL:g}) -> discarding the partial cache")
                _quarantine(out, f"[{arm}/{seed}] {tag}: partial cache written by other weights")
                have, todo_now = [], list(ids)
    else:                                       # "match"
        print(f"[{arm}/{seed}] {tag}: model identity: the {len(have)} cached map(s) MATCH the "
              f"attached checkpoint ({arm} {_fp_txt(mine)}); the remaining {len(todo_now)} "
              f"case(s) are computed from it here")

    os.makedirs(out, exist_ok=True)
    json.dump(_stamp([arm]), open(os.path.join(out, "_provenance.json"), "w"))
    model = bratsx._segresnet().to(device)
    st = torch.load(CKPTS[arm][seed], map_location=device, weights_only=False)
    model.load_state_dict(st["model"]); model.eval()
    todo = [p for p in ids if not os.path.exists(os.path.join(out, f"{p}.npz"))]
    print(f"[{arm}/{seed}] {tag}: {len(ids)-len(todo)} cached, {len(todo)} to compute")
    t0 = time.time()
    for i, pid in enumerate(todo):
        x, _, _ = bratsx.loader_case(CACHE_BASE, pid)
        pr = bratsx.infer_probs_tta(model, x, PATCH, INFER_OVERLAP, device, flips=TTA_FLIPS,
                                    tag=f"{arm}/seed{seed}, case {pid}")
        np.savez_compressed(os.path.join(out, f"{pid}.npz"),
                            p_wt=pr[0], p_tc=pr[1], p_et=pr[2])
        if (i + 1) % 10 == 0:
            print(f"   {i+1}/{len(todo)}  ({time.time()-t0:.0f}s)", flush=True)
    if todo:
        dt = time.time() - t0
        n_flip = max(1, len(TTA_FLIPS or [()]))
        print(f"[{arm}/{seed}] {tag}: {len(todo)} case(s) in {dt:.1f}s "
              f"= {dt/len(todo):.2f} s/case ({n_flip} pass(es) per case, TTA)")
    return out


PROB_DIRS = {}


def _seed_for(arm):
    """Which seed this notebook certifies: the first checkpoint seed if the runs are attached,
    otherwise the seed written into the cached directory name (`base_seed42`)."""
    if CKPTS.get(arm):
        return sorted(CKPTS[arm])[0]
    for d in sorted(glob.glob(os.path.join(PROBS, f"{arm}_seed*"))):
        if glob.glob(os.path.join(d, "*.npz")):
            return os.path.basename(d).rpartition("_seed")[2]
    return None


def _n2_check_2():
    import inspect, shutil
    print("=" * 88)
    print("CHECK 2/3  the corrected cells are the ones installed -- before the expensive stage")
    print("=" * 88)
    _allow = bool(globals().get("N2_ALLOW_UNPATCHED", False))
    ok = True
    fn = globals().get("infer_all")
    if fn is None:
        print("  FAIL    infer_all is not defined -- run the cell that defines it first")
        ok = False
    else:
        _src = inspect.getsource(fn)
        for _what, _mark in (("partial-cache identity check", "partial cache written by other "
                                                               "weights"),
                             ("identity probe print", "PROBED on")):
            _hit = _mark in _src
            print(f"  {'ok  ' if _hit else 'FAIL'}    cell 5 carries the {_what}")
            ok = ok and _hit
        if not ok and not _allow:
            raise RuntimeError(
                "cell 5 is the UNCORRECTED version.  With a PARTIAL `probs` cache attached this "
                "session can average two different models into one arm and stamp the result as "
                "fresh -- silently.  Paste the corrected cell 5 with the inserted block (see "
                "N2_FIX_cell5_insert.py), or set N2_ALLOW_UNPATCHED = True in cell 2 to run anyway.")
        if not ok:
            print("  NOTE    running unpatched on purpose (N2_ALLOW_UNPATCHED = True)")
    _fp = globals().get("CKPT_FP") or {}
    if _fp:
        print("  best.pt ", " | ".join(
            f"{a} selected_it={d['it']} best={d['best']} sha1={d['sha1']}"
            for a, d in sorted(_fp.items())))
        _progress = globals().get("TRAINING_FP") or {}
        _missing = [a for a in sorted(_fp) if a not in _progress]
        if _missing:
            print(f"  FAIL    no N1 completed-iteration metadata for: {_missing}")
            raise RuntimeError(
                "cannot verify the arms' completed N1 training budgets. best.pt stores the "
                "validation-selected snapshot, whose saved iteration may differ by arm; attach "
                "the full N1 output with its DONE JSON or last.pt and re-run cell 4.")
        _its = {a: int(_progress[a]["iters"]) for a in sorted(_fp)}
        print("  N1 done " + " | ".join(
            f"{a}/seed{_progress[a]['seed']} iters={_progress[a]['iters']} "
            f"[{_progress[a]['source']}]" for a in sorted(_fp)))
        if len(set(_its.values())) > 1:
            print(f"  FAIL    completed N1 budgets differ: {_its}")
            raise RuntimeError(
                "the arms completed different total N1 iteration budgets. This check uses DONE/"
                "last.pt, not the per-arm validation-selected step in best.pt. Finish the short arm "
                "in N1 before certifying.")
        print("  ok      N1 completed budgets match; best.pt selection steps may differ legitimately")
    else:
        print("  fp      none -- this session certifies from a `probs` dataset, no model is built")
    for _p in [x for x in {globals().get("WORK"), "/kaggle/working"} if x]:
        try:
            _free = shutil.disk_usage(_p).free / 2**30
            print(f"  disk    {_free:6.1f} GB free in {_p}")
            if _free < 10.0:
                print("  WARN    under 10 GB: the per-case probability maps are the big artefact "
                      "of this stage")
        except Exception:
            pass
    print("  note    long GPU minutes expected: 2 arms x (37 val + 37 test) cases, cached by case; "
          "a re-run of this cell is cheap when nothing is missing")
    print("CHECK 2/3 done")


_n2_check_2()

# use the first seed of each arm for the report; every seed is kept on disk
SEED0 = None
_short = []
for arm in ARMS:                    # EXPERIMENT arms only; derived ones are built just below
    seed0 = _seed_for(arm)
    if seed0 is None:
        _short.append(arm)
        continue
    SEED0 = seed0 if SEED0 is None else SEED0
    PROB_DIRS[arm] = infer_all(arm, seed0, VAL_IDS, "val")
    if RUN_TEST:
        infer_all(arm, seed0, TEST_IDS, "test")
assert not _short, (
    f"nothing to certify from for {_short}: no checkpoint under {RUNS} and no cached "
    f"probabilities for these arms.  Attach the N1 runs output, or a `probs` dataset containing "
    f"<arm>_seed<k>/<case_id>.npz for all {len(VAL_IDS)} validation cases"
    + (f" and {len(TEST_IDS)} test cases" if RUN_TEST else "") + ".")

# ---- derived probability maps: the ensemble of the arms, from the cached maps --------------
# Averaging happens on the PROBABILITIES, before any threshold -- which is where averaging is
# meaningful.  No extra inference and no new weights: the same models voting.
def build_ensembles(case_ids):
    made = []
    for name, srcs in (ENSEMBLE or []):
        srcs = [a for a in srcs if os.path.isdir(PROB_DIRS.get(a, ""))]
        if len(srcs) < 2:
            print(f"[{name}] skipped: needs >= 2 arms with cached probabilities, found {srcs}")
            continue
        dst = os.path.join(PROBS, f"{name}_seed{SEED0}")
        os.makedirs(dst, exist_ok=True)
        # An ensemble is only as good as its sources: if an arm was recomputed in this session
        # (a different checkpoint, a different recipe), an ensemble averaged from the attached
        # cache would silently mix the two.  Reuse only when the stamp says it was built from
        # exactly these maps; otherwise average again -- it is numpy arithmetic, seconds of CPU.
        want = dict(ensemble=list(srcs), flips=_provenance()["flips"])
        if any(a in CKPT_FP for a in srcs):
            want["ckpt"] = {a: CKPT_FP[a] for a in srcs if a in CKPT_FP}
        have = None
        try:
            have = json.load(open(os.path.join(dst, "_provenance.json")))
        except Exception:
            have = None
        if have != want:
            _old = glob.glob(os.path.join(dst, "*.npz"))
            if _old:
                print(f"[{name}] its cached maps were built from different inputs -> averaging the "
                      f"current ones ({len(_old)} file(s) replaced)")
            for _f in _old:
                os.remove(_f)
        n_new = 0
        for pid in case_ids:
            if not all(os.path.exists(os.path.join(PROB_DIRS[a], f"{pid}.npz")) for a in srcs):
                continue
            o = os.path.join(dst, f"{pid}.npz")
            if os.path.exists(o):
                continue
            acc = {}
            for a in srcs:
                with np.load(os.path.join(PROB_DIRS[a], f"{pid}.npz")) as z:
                    for k in ("p_wt", "p_tc", "p_et"):
                        acc[k] = acc.get(k, 0) + z[k].astype(np.float32)
            np.savez_compressed(o, **{k: (v / len(srcs)).astype(np.float16)
                                      for k, v in acc.items()})
            n_new += 1
        json.dump(want, open(os.path.join(dst, "_provenance.json"), "w"))
        PROB_DIRS[name] = dst
        made.append(name)
        print(f"[{name}] ensemble of {srcs}: {n_new} new, {len(case_ids) - n_new} reused "
              f"(no extra inference)")
    return made


DERIVED = build_ensembles(VAL_IDS + (TEST_IDS if RUN_TEST else []))
if DERIVED:
    ARMS = list(ARMS) + DERIVED      # every downstream pass picks the derived arms up
    print(f"arms in this report: {ARMS}  (derived: {DERIVED})")


# ---- cohort audit: every probability map the certificate will stand on must be finite ----
# This also covers files cached by an EARLIER session (the loop above skips existing files), i.e.
# a probs dataset published by a build that certified a poisoned checkpoint.
def audit_probs(ids, tag):
    bad = {}
    for arm in ARMS:
        d = PROB_DIRS[arm]
        hits = []
        for pid in ids:
            f = os.path.join(d, f"{pid}.npz")
            if not os.path.exists(f):
                hits.append((pid, "missing"))
                continue
            with np.load(f) as z:
                if any(not np.isfinite(z[k]).all() for k in ("p_wt", "p_tc", "p_et")):
                    hits.append((pid, "non-finite"))
        if hits:
            bad[arm] = hits
    if not bad:
        print(f"cohort audit [{tag}]: every probability map is finite ({len(ids)} cases x "
              f"{len(ARMS)} arm(s))")
        return ids
    for arm, hits in bad.items():
        print(f"!! [{arm}] {tag}: {len(hits)} unusable case(s): "
              f"{[(p, why) for p, why in hits[:6]]}{' …' if len(hits) > 6 else ''}")
    _msg = (f"{sum(len(v) for v in bad.values())} probability map(s) are missing or non-finite.\n"
            f"  A non-finite map compares as an EMPTY prediction: the BraTS convention scores that "
            f"0.0 (or 1.0 when the reference region is empty), so a dead checkpoint would be "
            f"certified and reported as a merely weak one -- in silence.  That is why this stops.\n"
            f"  Cause: the checkpoint is poisoned, i.e. written by a run whose weights went nan "
            f"(a build without the GradScaler).  Fix: re-run N1 -- it detects nan weights on "
            f"resume, moves that run aside and retrains the arm; code/check_run.py flags the "
            f"non-finite loss in train_log.csv.\n"
            f"  Deliberate override: ALLOW_NONFINITE_CASES=True drops those cases from the cohort "
            f"(the report then says how many) instead of stopping.  Read what that does to n_cal "
            f"before using it.")
    if not ALLOW_NONFINITE_CASES:
        raise RuntimeError(_msg)
    _drop = sorted({p for hits in bad.values() for p, _ in hits})
    print("!! ALLOW_NONFINITE_CASES=True -> dropping " + str(len(_drop)) + " case(s) from the "
          "cohort: " + str(_drop) + "\n   " + _msg.replace(chr(10), chr(10) + "   "))
    json.dump(_drop, open(os.path.join(WORK, "nonfinite_cases.json"), "w"), indent=1)
    return [p for p in ids if p not in set(_drop)]


VAL_IDS = audit_probs(VAL_IDS, "val")
if RUN_TEST:
    TEST_IDS = audit_probs(TEST_IDS, "test")

# ---- sanity: the probabilities have to look like a trained model ------------------------
# A raw threshold-0.5 mask from a trained network cannot be worse than "predict nothing"; if it
# is, the INFERENCE is broken, not the model.  This exact failure happened once: a wrong axis in
# the flip-averaging un-flip swapped the WT/TC/ET maps and averaged three inconsistent volumes,
# which turned 0.84 into 0.03 without raising anything.  So: print the per-region split, and stop
# the run rather than certify a mask that is known to be nonsense.
VAL_SCORE_FLOOR = 0.30
_sane = {}
for arm in ARMS:
    d = PROB_DIRS[arm]
    per_region, scores = {"WT": [], "TC": [], "ET": []}, []
    for pid in VAL_IDS:
        z = np.load(os.path.join(d, f"{pid}.npz"))
        raw = safepp.labels_from_probs(z["p_wt"], z["p_tc"], z["p_et"], 0.5)
        seg = np.load(bratsx.cache_file(CACHE_BASE, pid, "seg"))
        dr = safepp.dice_regions(raw.astype(np.uint8), seg)
        for k in per_region:
            per_region[k].append(dr[k])
        scores.append(np.mean(list(dr.values())))
        # a probability map that is constant, or saturated, cannot produce a usable mask
        if not (0.001 < float(z["p_wt"].mean()) < 0.999):
            print(f"!! [{arm}] {pid}: mean p_wt = {float(z['p_wt'].mean()):.4f} -- "
                  f"the probability map is (near-)constant")
    mean_score = float(np.mean(scores))
    _sane[arm] = mean_score
    print(f"[{arm}] val mean Dice @0.5 (raw, no post-processing): {mean_score:.4f}"
          f"   | WT {np.mean(per_region['WT']):.3f}  TC {np.mean(per_region['TC']):.3f}"
          f"  ET {np.mean(per_region['ET']):.3f}")

_best = max(_sane.values())
if _best < VAL_SCORE_FLOOR:
    raise RuntimeError(
        f"every arm scores below {VAL_SCORE_FLOOR} mean Dice at threshold 0.5 (best: "
        f"{_best:.4f}, {_sane}).  The probabilities are not from a functioning model, so the "
        f"certificate would certify noise.  Check, in this order:\n"
        f"  1. the inference recipe -- `TTA_FLIPS` with the current build; run "
        f"`python3 code/test_tta_flips.py` locally, or set TTA_FLIPS = [] to fall back to plain "
        f"inference and re-run;\n"
        f"  2. whether the cached maps were written by an older build (the log line above "
        f"'moved to ...__stale and recomputed' means they were not reused, which is correct);\n"
        f"  3. whether the checkpoints match the cache -- run `python3 code/check_run.py` and "
        f"compare the build line in the run header with this notebook's.")
print(f"sanity: best arm scores {_best:.4f} at threshold 0.5 (floor {VAL_SCORE_FLOOR}) -> the "
      f"probabilities look like a trained model")


[probs] no cached probability maps found: attach a `probs` dataset (Add Data -> the dataset made from an earlier N2 output) or the N1 runs, or this session has to run inference and needs the checkpoints.
CHECK 2/3  the corrected cells are the ones installed -- before the expensive stage
  ok      cell 5 carries the partial-cache identity check
  ok      cell 5 carries the identity probe print
  best.pt  base selected_it=19500 best=0.8191 sha1=418a2c28a16c | sr50 selected_it=12500 best=0.8167 sha1=24298f8183bc
  N1 done base/seed42 iters=21795 [DONE] | sr50/seed42 iters=21795 [DONE]
  ok      N1 completed budgets match; best.pt selection steps may differ legitimately
  disk      19.5 GB free in /kaggle/working
  note    long GPU minutes expected: 2 arms x (37 val + 37 test) cases, cached by case; a re-run of this cell is cheap when nothing is missing
CHECK 2/3 done
[base/42] val: 0 cached, 37 to compute
   10/37  (54s)
   20/37  (108s)
   30/37  (164s)
[base/42] val: 37 case(s) in 200.8

KeyboardInterrupt: 

In [ ]:
# %% [CELL 6] TUNE POST-PROCESSING ON VALIDATION ONLY, THEN FREEZE ------------
def apply_pipeline(probs_npz, lam, params):
    thr = safepp.region_thresholds(lam, params)
    raw = safepp.labels_from_probs(probs_npz["p_wt"], probs_npz["p_tc"], probs_npz["p_et"], thr)
    return raw, safepp.postproc_seg(raw, params)


def _tune_arm():
    """Which arm's probabilities the post-processing is tuned on.

    "auto" prefers the delivered configuration (the first ARM_PREFERENCE entry that exists).
    The constraint that matters is untouched either way: exactly ONE parameter set, tuned on
    VALIDATION, is applied to every arm, so the arms stay comparable.
    """
    if POST_TUNE_ARM and POST_TUNE_ARM != "auto":
        return POST_TUNE_ARM if POST_TUNE_ARM in PROB_DIRS else ARMS[0]
    for a in (ARM_PREFERENCE or []):
        if a in PROB_DIRS:
            return a
    return ARMS[0]


def _post_score(post_seg, gt_seg, lw_gt=None):
    """The objective post-processing is tuned for: accuracy, fragmentation, and honesty about
    volume -- not mean voxel Dice alone.

    Voxel Dice rises when a prediction swallows more of the volume, so a filter that deletes a
    real satellite can still look good on it (that is exactly how a 400-voxel floor survived
    tuning in an earlier run).  Lesion-wise Dice charges a missed satellite in full, and the
    volume term charges over-inflation directly.  Weights live in POST_OBJ_W.
    """
    w = POST_OBJ_W or {}
    d = safepp.dice_regions(post_seg, gt_seg)
    m_dice = float(np.mean(list(d.values())))
    lw, _ = safepp.lesionwise_dice(post_seg, gt_seg, PRIMARY.upper(), gt_labels=lw_gt)
    m_lw = float(lw) if np.isfinite(lw) else 1.0
    g = int(np.isin(gt_seg, safepp.REGIONS[PRIMARY.upper()]).sum())
    p = int(np.isin(post_seg, safepp.REGIONS[PRIMARY.upper()]).sum())
    m_vol = min(1.0, abs(p - g) / max(1, g))
    return (float(w.get("dice", 1.0)) * m_dice + float(w.get("lesion", 0.0)) * m_lw
            - float(w.get("volume", 0.0)) * m_vol), dict(dice=m_dice, lesion=m_lw, volume=m_vol)


def tune_post(arm):
    """Grid-search exactly one parameter set for ALL arms (a per-arm setting would make the
    arms incomparable).  Objective: `_post_score`.  VALIDATION only.

    Three stages:
      1. component-size floor + ET-to-necrosis rule at a nominal threshold;
      2. per-region TC/ET thresholds, with WT left for the certificate to pick;
      3. (CERTIFIED_FILTER) the floor itself from the certified filter -- the largest component
         size whose collateral damage on real lesions is risk-controlled on validation -- which
         replaces a search that was free to prefer a floor that deletes satellites.
    """
    tune_arm = _tune_arm()
    print(f"  post-processing tuned on [{tune_arm}] probabilities (validation only; the same "
          f"parameters are applied to every arm)")
    val_probs = {pid: np.load(os.path.join(PROB_DIRS[tune_arm], f"{pid}.npz")) for pid in VAL_IDS}
    val_seg = {pid: np.load(bratsx.cache_file(CACHE_BASE, pid, "seg")) for pid in VAL_IDS}
    # label the ground truth ONCE: the sweep calls the lesion-wise metric thousands of times
    lw_gt = {pid: safepp.components(np.isin(val_seg[pid], safepp.REGIONS[PRIMARY.upper()]))
             for pid in VAL_IDS}

    rows = []
    best, best_score = None, -1e9
    ref_lam = float(np.median(list(POST_THR_GRID)))                  # WT's mask matters only here
    for thr in POST_THR_GRID:
        for cc in POST_CC_GRID:
            for et in POST_ET_GRID:
                params = dict(min_vox={"WT": cc, "TC": cc, "ET": cc}, min_et_voxels=et)
                sc, parts = [], {"dice": [], "lesion": [], "volume": []}
                for pid in VAL_IDS:
                    raw = safepp.labels_from_probs(val_probs[pid]["p_wt"], val_probs[pid]["p_tc"],
                                                   val_probs[pid]["p_et"], thr)
                    post = safepp.postproc_seg(raw, params)
                    m, comp = _post_score(post, val_seg[pid], lw_gt[pid])
                    sc.append(m)
                    for k in parts:
                        parts[k].append(comp[k])
                m = float(np.mean(sc))
                rows.append(dict(stage=1, thr=thr, cc=cc, et=et, objective=m,
                                 **{k: float(np.mean(v)) for k, v in parts.items()}))
                if m > best_score:
                    best_score, best = m, params

    # stage 2: per-region TC/ET thresholds (WT is the certified lambda, never tuned here)
    if POST_REGION_THR_GRID:
        t_best, t_score = (None, None), -1e9
        for tc_t in POST_REGION_THR_GRID:
            for et_t in POST_REGION_THR_GRID:
                sc, parts = [], {"dice": [], "lesion": [], "volume": []}
                for pid in VAL_IDS:
                    params = dict(best, thr_tc=tc_t, thr_et=et_t)
                    _, post = apply_pipeline(val_probs[pid], ref_lam, params)
                    m, comp = _post_score(post, val_seg[pid], lw_gt[pid])
                    sc.append(m)
                    for k in parts:
                        parts[k].append(comp[k])
                m = float(np.mean(sc))
                rows.append(dict(stage=2, thr=ref_lam, cc=best["min_vox"]["WT"],
                                 et=best["min_et_voxels"], thr_tc=tc_t, thr_et=et_t,
                                 objective=m, **{k: float(np.mean(v)) for k, v in parts.items()}))
                if m > t_score:
                    t_score, t_best = m, (float(tc_t), float(et_t))
        if t_best[0] is not None:
            best = dict(best, thr_tc=t_best[0], thr_et=t_best[1])
            print(f"  stage 2: per-region thresholds TC {t_best[0]:.2f}, ET {t_best[1]:.2f} "
                  f"(validation objective {t_score:.4f})")

    # stage 3: the component floor from the certified filter, per region
    if CERTIFIED_FILTER:
        deltas, notes = {}, []
        for reg in ("WT", "TC", "ET"):
            curves = []
            for pid in VAL_IDS:
                params = dict(best, min_vox={"WT": 1, "TC": 1, "ET": 1})
                _, post1 = apply_pipeline(val_probs[pid], ref_lam, params)
                curves.append(safepp.removal_curve(post1, val_seg[pid],
                                                   np.array(FILTER_GRID, float), region=reg))
            f = safepp.certified_filter_threshold(np.stack(curves), alpha=float(
                (FILTER_ALPHAS or [0.05])[0]), thresholds=np.array(FILTER_GRID, float),
                component_size_floor=min(FILTER_GRID[-2], 300.0))
            deltas[reg] = int(f["threshold"])
            notes.append(f"{reg} delta={deltas[reg]} (bound {f['bound_cal']:.3f})")
        best = dict(best, min_vox=deltas)
        print("  stage 3: certified component floors: " + " | ".join(notes))
        print("           (largest size whose collateral damage on real lesions is "
              "risk-controlled on validation; the old grid search was free to prefer a floor "
              "that deletes satellites)")
    else:
        print(f"  stage 3: CERTIFIED_FILTER is off -- component floor kept at {best['min_vox']}")

    return best, float(best_score), pd.DataFrame(rows)


if FORCE_POST:
    POST = dict(FORCE_POST); print("post-processing FORCED (not tuned):", POST)
else:
    POST, val_score, grid = tune_post(ARMS[0])
    # post_grid.csv keeps EVERY candidate with all three objective components, so the trade-off
    # is auditable rather than a black box that handed back one number
    grid.sort_values("objective", ascending=False).head(20).to_csv(
        os.path.join(WORK, "post_grid.csv"), index=False)
    print(f"post-processing tuned on {len(VAL_IDS)} val cases: {POST}"
          f"\n  validation objective {val_score:.4f} "
          f"(weights {POST_OBJ_W}; components in post_grid.csv)")
json.dump(POST, open(os.path.join(WORK, "frozen_postproc.json"), "w"), indent=1)


In [ ]:
# %% [CELL 7] CALIBRATE THE CERTIFICATES ON VALIDATION ------------------------
# CORRECTED BUILD: + per-case calibration cache (curve_risk); nothing else changed.
def risk_curves(arm, ids):
    """Per-case missed-tumour and wrong-prediction curves for the DELIVERED pipeline
    (threshold + post-processing), because that is what the certificate is about."""
    FN, FP = [], []
    for pid in ids:
        f_, p_ = curve_risk(arm, pid)
        FN.append(f_); FP.append(p_)
    return (safepp.monotone_envelope(np.array(FN), "increasing"),
            safepp.monotone_envelope(np.array(FP), "decreasing"))


# Per-case cache.  ONE case costs |LAMBDAS| x |TEST_IDS| passes through apply_pipeline, so the
# pooled curves are the single most expensive computation in the notebook, and BOTH risk curves and
# the stratified ones (1 + 2 x n_strata rebuilds, the strata being subsets of the pooled set) used
# to pay for the same cases over and over.  Keyed by (arm, case), since the curve depends on POST,
# which is frozen before this cell runs.
CURVE_CACHE = {}


def curve_risk(arm, pid):
    """(missed-fraction curve, wrong-fraction curve) for one case at the frozen POST."""
    key = (arm, pid)
    if key not in CURVE_CACHE:
        reg = PRIMARY.upper()
        z = np.load(os.path.join(PROB_DIRS[arm], f"{pid}.npz"))
        seg = np.load(bratsx.cache_file(CACHE_BASE, pid, "seg"))
        f_, p_ = [], []
        for lam in LAMBDAS:
            raw, post = apply_pipeline(z, lam, POST)
            pred = np.isin(post, safepp.REGIONS[reg])
            gt = np.isin(seg, safepp.REGIONS[reg])
            f_.append(0.0 if gt.sum() == 0 else float((gt & ~pred).sum()) / int(gt.sum()))
            p_.append(0.0 if pred.sum() == 0 else float((pred & ~gt).sum()) / int(pred.sum()))
        CURVE_CACHE[key] = (f_, p_)
    return CURVE_CACHE[key]


CERT = {}
VALID_CAL = len(VAL_IDS) >= MIN_CAL
if not VALID_CAL:
    print(f"!! only {len(VAL_IDS)} calibration case(s), MIN_CAL={MIN_CAL}.  The finite-sample "
          f"bound is (n/(n+1))*mean + 1/(n+1), which at n={len(VAL_IDS)} is dominated by the "
          f"+1/(n+1) term: the numbers below are ILLUSTRATIVE, not a certificate.\n"
          f"   Attach the full split (369 cases -> ~74 validation cases) before quoting any "
          f"coverage number.")
print(f"calibrating on {len(VAL_IDS)} validation cases (region {PRIMARY.upper()}) ...")
GRIDS, CURVES = {}, {}
for arm in ARMS:
    FN, FP = risk_curves(arm, VAL_IDS)
    CURVES[arm] = (FN, FP)
    GRIDS[arm] = {}
    for a in ALPHAS:
        for b in BETAS:
            c = safepp.certified_threshold_pair(FN, FP, LAMBDAS, alpha=a, beta=b)
            GRIDS[arm][(float(a), float(b))] = dict(lam=c["lam"], feasible=c["feasible"],
                                                    thr_fn=c["thr_fn"], thr_fp=c["thr_fp"])

# ONE operating point for every arm.  (alpha, beta) is the promise; each arm then runs at its own
# tightest threshold that keeps it -- so two arms certified at DIFFERENT cells are not being
# compared on the same promise, and "all arms identical except the input channels" stops being
# true.  Rule, fixed before looking at test: the pre-registered cell (smallest alpha, largest
# beta) if EVERY arm can meet it; otherwise the first cell -- alpha ascending, then beta -- that
# every arm can meet; if no such cell exists, each arm falls back to its own first feasible cell
# and the report says, in as many words, that the arms are not at a common operating point.
PRE_A, PRE_B = float(ALPHAS[0]), float(BETAS[-1])
CELL, CELL_SHARED = None, True
if FORCE_ALPHA is not None or FORCE_BETA is not None:
    CELL = (float(FORCE_ALPHA) if FORCE_ALPHA is not None else PRE_A,
            float(FORCE_BETA) if FORCE_BETA is not None else PRE_B)
    print(f"  operating point PINNED by FORCE_* to alpha={CELL[0]:.2f}, beta={CELL[1]:.2f}")
else:
    for (a, b) in [(float(a), float(b)) for a in ALPHAS for b in BETAS]:
        if all(GRIDS[arm][(a, b)]["feasible"] for arm in ARMS):
            CELL = (a, b); break
    if CELL is None:
        CELL_SHARED = False
        print("  !! NO (alpha, beta) cell on the grid is feasible for every arm: the arms below "
              "are certified at their own cells and are NOT at a common operating point")
    elif CELL != (PRE_A, PRE_B):
        print(f"  !! the pre-registered cell (alpha={PRE_A:.2f}, beta={PRE_B:.2f}) is not feasible "
              f"for every arm; using (alpha={CELL[0]:.2f}, beta={CELL[1]:.2f}), which is")

for arm in ARMS:
    grid = GRIDS[arm]
    if CELL is not None:
        a, b = CELL
        chosen = grid[(a, b)]
        note = None if (a, b) == (PRE_A, PRE_B) else (
            f"the pre-registered cell (alpha={PRE_A:.2f}, beta={PRE_B:.2f}) is not feasible for "
            f"every arm; this run uses (alpha={a:.2f}, beta={b:.2f}), which is -- recorded in "
            f"certificates.json and shown on the report page")
    else:
        a, b = next(((ka, kb) for (ka, kb), g in grid.items() if g["feasible"]),
                    (min(ALPHAS), max(BETAS)))
        chosen = grid[(a, b)]
        note = ("no cell is feasible for every arm, so this arm is certified at its own cell: "
                "the arm-to-arm comparison is NOT at a common operating point")
    FN, FP = CURVES[arm]
    _n = len(VAL_IDS)
    CERT[arm] = dict(alpha=float(a), beta=float(b), n_cal=_n, region=PRIMARY.upper(),
                     valid=bool(VALID_CAL),
                     warning=(None if VALID_CAL else
                              f"only {_n} calibration cases (MIN_CAL={MIN_CAL}); "
                              f"treat every number in this report as illustrative"),
                     cell_shared=bool(CELL_SHARED), cell_note=note,
                     curves=dict(
        fn=FN.mean(axis=0).tolist(), fp=FP.mean(axis=0).tolist(),
        fn_bound=((_n / (_n + 1)) * FN.mean(axis=0) + 1 / (_n + 1)).tolist(),
        fp_bound=((_n / (_n + 1)) * FP.mean(axis=0) + 1 / (_n + 1)).tolist(),
        lams=list(map(float, LAMBDAS))),
        grid={f"{x:.2f}|{y:.2f}": g for (x, y), g in grid.items()},
        **chosen)
    # the bug this replaces: alpha/beta were recorded from the PRE-REGISTERED cell while lambda*
    # came from whatever cell was feasible, so the risk check, the volume interval and the
    # coverage number were all computed at a promise the mask never met.
    assert abs(CERT[arm]["alpha"] - a) < 1e-12 and abs(CERT[arm]["beta"] - b) < 1e-12, \
        "the recorded alpha/beta must be the cell that produced lambda*"
    print(f"  [{arm}] alpha={a:.2f} beta={b:.2f} -> lambda*={chosen['lam']} "
          f"feasible={chosen['feasible']} (band [{chosen['thr_fp']}, {chosen['thr_fn']}])")
    if note:
        print(f"       note: {note}")

# stratified certificates: the guarantee inside the low-contrast subgroup
if len(man) and "wt_cnr" in man.columns:
    cut = float(man.loc[[p for p in VAL_IDS if p in man.index], "wt_cnr"].quantile(1 / 3))
    for arm in ARMS:
        low = [p for p in VAL_IDS if p in man.index and man.loc[p, "wt_cnr"] <= cut]
        high = [p for p in VAL_IDS if p in man.index and man.loc[p, "wt_cnr"] > cut]
        if len(low) >= MIN_STRATUM_N and len(high) >= MIN_STRATUM_N:
            def _curve_pair(ids):
                # reuse the per-case cache: a stratum is a SUBSET of the cases the pooled curve
                # already paid for, so this line costs almost nothing
                FN, FP = risk_curves(arm, ids)
                return dict(fn=FN.mean(axis=0).tolist(), fp=FP.mean(axis=0).tolist())
            FNl, FPl = risk_curves(arm, low); FNh, FPh = risk_curves(arm, high)
            cl = safepp.certified_threshold_pair(FNl, FPl, LAMBDAS, CERT[arm]["alpha"], CERT[arm]["beta"])
            ch = safepp.certified_threshold_pair(FNh, FPh, LAMBDAS, CERT[arm]["alpha"], CERT[arm]["beta"])
            CERT[arm]["strata"] = dict(
                cut=cut, n_low=len(low), n_high=len(high),
                lam_low=cl["lam"], lam_high=ch["lam"],
                feasible_low=cl["feasible"], feasible_high=ch["feasible"],
                thr_fn_low=cl["thr_fn"], thr_fn_high=ch["thr_fn"],
                curves_low=_curve_pair(low), curves_high=_curve_pair(high))
            print(f"  [{arm}] stratified: low-contrast (n={len(low)}) lambda={cl['lam']} "
                  f"feasible={cl['feasible']} vs high (n={len(high)}) lambda={ch['lam']} "
                  f"feasible={ch['feasible']}")
        else:
            print(f"  [{arm}] stratification skipped: only {len(low)}/{len(high)} val cases per "
                  f"side (MIN_STRATUM_N={MIN_STRATUM_N}) -> the pooled certificate is used")

# the case -> stratum assignment, written once: N3 needs it and it must not be recomputed there
if len(man) and "wt_cnr" in man.columns and "strata" in CERT[ARMS[0]]:
    _cut = CERT[ARMS[0]]["strata"]["cut"]
    _stable = [p for p in (VAL_IDS + TEST_IDS) if p in man.index]
    pd.DataFrame([dict(pid=p, wt_cnr=float(man.loc[p, "wt_cnr"]),
                       stratum=("low" if float(man.loc[p, "wt_cnr"]) <= _cut else "high"))
                  for p in _stable]).to_csv(os.path.join(WORK, "strata.csv"), index=False)
    print(f"strata.csv written for {len(_stable)} val/test cases (cut {_cut:.2f})")
json.dump(CERT, open(os.path.join(WORK, "certificates.json"), "w"), indent=1)


In [ ]:
# %% [CELL 8] TEST -- touched once, after everything is frozen ----------------
# CORRECTED BUILD: the per-case cache is now keyed by (arm, lambda) only.  See `eval_all`.
def _n2_check_3():
    import shutil
    print("=" * 88)
    print("CHECK 3/3  calibration is frozen -- holds before the test touch")
    print("=" * 88)
    _allow = bool(globals().get("N2_ALLOW_UNPATCHED", False))
    _fails = []
    if "CKPT_FP" not in globals() and "VAL_IDS" not in globals():
        _fails.append("cell 4 has not run (no CKPT_FP / VAL_IDS): run cell 4 first")
    if "CURVE_CACHE" not in globals():
        _fails.append("cell 7 is the UNCORRECTED version (no CURVE_CACHE): the strata rebuild "
                      "every pooled curve -- ~3 500 instead of ~2 100 apply_pipeline passes per arm")
    else:
        _cc = globals().get("CURVE_CACHE") or {}
        _per = {}
        for _k in _cc:
            _a = _k[0] if isinstance(_k, tuple) else _k
            _per[_a] = _per.get(_a, 0) + 1
        print("  curves  " + (", ".join(f"{a}: {n} case(s)" for a, n in sorted(_per.items()))
                              or "empty -- calibration has not run yet"))
    _cert = globals().get("CERT") or {}
    for _a, _c in sorted(_cert.items()):
        print(f"  cert    {_a}: alpha={_c.get('alpha')} beta={_c.get('beta')} "
              f"lam*={_c.get('lam')} n_cal={_c.get('n_cal')} feasible={_c.get('feasible')}")
    if not _cert:
        _fails.append("CERT is empty: run the calibration cell (or check its '!!' lines)")
    for _p in [x for x in {globals().get("WORK"), "/kaggle/working"} if x]:
        try:
            print(f"  disk    {shutil.disk_usage(_p).free / 2**30:6.1f} GB free in {_p}")
        except Exception:
            pass
    print("  expect  cell 8 -> per-(arm, lambda) rows in EVAL_CACHE; the 9 (alpha, beta) cells "
          "collapse to ~3 distinct thresholds per arm")
    print("  expect  cell 9 -> promise columns rebuilt per row by promise_columns; the coverage "
          "table prints with standard errors; only `reflected` rows describe the delivered mask")
    if _fails:
        for _f in _fails:
            print(f"  FAIL    {_f}")
        if not _allow:
            raise RuntimeError("CHECK 3/3 failed: " + "; ".join(_fails))
    print("CHECK 3/3 done")


_n2_check_3()

def evaluate(arm, ids, lam, params):
    """All metrics the report and the paper table need, per case.

    The volume-interval COLUMNS in this frame are the ARM'S certified promise [v(1-beta), v/(1-alpha)].
    A row that is about a different (alpha, beta) must have them rebuilt, which is what
    `promise_columns(df, alpha, beta)` in the analysis cell is for -- the interval is a property of
    the promise, not of the threshold, so every (alpha, beta) cell sharing one lambda would otherwise
    report the same interval and its "interval contained truth" column would answer a question the
    header does not ask.
    """
    reg = PRIMARY.upper()
    rows = []
    for pid in ids:
        z = np.load(os.path.join(PROB_DIRS[arm], f"{pid}.npz"))
        seg = np.load(bratsx.cache_file(CACHE_BASE, pid, "seg"))
        raw, post = apply_pipeline(z, lam, params)
        d_post = safepp.dice_regions(post, seg)
        d_raw = safepp.dice_regions(raw, seg)
        pred = np.isin(post, safepp.REGIONS[reg]); gt = np.isin(seg, safepp.REGIONS[reg])
        a_here = float(CERT[arm]["alpha"])
        b_here = float(CERT[arm]["beta"])
        lo, hi = safepp.volume_interval(float(pred.sum()), a_here, b_here)
        st = safepp.component_stats(post, seg, reg)
        st_raw = safepp.component_stats(raw, seg, reg)
        rows.append(dict(
            pid=pid, cnr=float(man.loc[pid, "wt_cnr"]) if (len(man) and pid in man.index) else np.nan,
            dice_wt=d_post["WT"], dice_tc=d_post["TC"], dice_et=d_post["ET"],
            dice_wt_raw=d_raw["WT"], dice_tc_raw=d_raw["TC"], dice_et_raw=d_raw["ET"],
            hd95_wt=safepp.hd95(post, seg, "WT"), hd95_et=safepp.hd95(post, seg, "ET"),
            lw_dice=safepp.lesionwise_dice(post, seg, reg)[0],
            # raw = BEFORE the component filter.  Without these two, a missed satellite cannot
            # be blamed on the model rather than on the >= 400-voxel floor that deleted it.
            lw_dice_raw=safepp.lesionwise_dice(raw, seg, reg)[0],
            missed_lesions_raw=st_raw["missed_lesions"], n_gt=st["n_gt"],
            et_gt_voxels=int(np.isin(seg, safepp.REGIONS["ET"]).sum()),
            et_pred_voxels=int(np.isin(post, safepp.REGIONS["ET"]).sum()),
            n_components=st["n_pred"], false_components=st["false_components"],
            missed_lesions=st["missed_lesions"], beta0_error=st["beta0_error"],
            n_components_raw=st_raw["n_pred"], false_components_raw=st_raw["false_components"],
            volume_pred=float(pred.sum()), volume_gt=float(gt.sum()),
            volume_err=float(pred.sum() - gt.sum()),
            volume_err_raw=float(np.isin(raw, safepp.REGIONS[reg]).sum() - gt.sum()),
            vol_lo=lo, vol_hi=hi, vol_covered=bool(lo <= float(gt.sum()) <= hi),
            missed_frac=0.0 if gt.sum() == 0 else float((gt & ~pred).sum()) / int(gt.sum()),
            wrong_frac=0.0 if pred.sum() == 0 else float((pred & ~gt).sum()) / int(pred.sum()),
            lam=lam))
    return pd.DataFrame(rows)


CSV_DIR = os.path.join(WORK, "test_csvs"); os.makedirs(CSV_DIR, exist_ok=True)

# One full pass over val+test per DISTINCT (arm, lambda), then every table/figure/stratum is a
# row selection.  The mask -- and therefore every column here except the promise ones -- is a
# function of (arm, lambda) ALONE: the (alpha, beta) grid asks 9 questions but they collapse to
# ~3 distinct thresholds per arm.  So the key must NOT carry alpha/beta, or the 18 passes the
# comment promises to avoid come straight back (the promise columns are rebuilt per cell by
# `promise_columns`, which is why this is safe).
EVAL_CACHE = {}
ALL_IDS = list(VAL_IDS) + list(TEST_IDS)


def eval_all(arm, lam):
    key = (arm, round(float(lam), 6))
    if key not in EVAL_CACHE:
        EVAL_CACHE[key] = evaluate(arm, ALL_IDS, lam, POST).set_index("pid")
    return EVAL_CACHE[key]


def eval_subset(arm, lam, ids):
    df = eval_all(arm, lam)
    return df.loc[[p for p in ids if p in df.index]].reset_index()


VAL_DF, TEST_DF = {}, {}
for arm in ARMS:
    lam = CERT[arm]["lam"]
    VAL_DF[arm] = eval_subset(arm, lam, VAL_IDS)
    VAL_DF[arm].to_csv(os.path.join(CSV_DIR, f"{arm}_val.csv"), index=False)
    if RUN_TEST:
        TEST_DF[arm] = eval_subset(arm, lam, TEST_IDS)
        TEST_DF[arm].to_csv(os.path.join(CSV_DIR, f"{arm}_test.csv"), index=False)
        print(f"[{arm}] lambda*={lam:.2f}  "
              f"test median Dice WT {TEST_DF[arm].dice_wt.median():.3f} | "
              f"missed {TEST_DF[arm].missed_frac.mean():.3f} | "
              f"wrong {TEST_DF[arm].wrong_frac.mean():.3f} | "
              f"interval coverage {TEST_DF[arm].vol_covered.mean():.0%}")

if RUN_TEST:
    key = TEST_DF[ARMS[0]]
    print("\nper-case table (first rows):")
    print(key[["pid", "cnr", "dice_wt", "dice_tc", "dice_et", "hd95_wt", "lw_dice",
               "false_components", "missed_frac", "wrong_frac", "vol_covered"]].head(8).round(3)
          .to_string(index=False))


In [ ]:
# %% [CELL 9] ANALYSIS -- does the certificate hold, and what did it cost? ----
def promise_columns(df, alpha, beta):
    """Attach the volume interval implied by THIS row's promise, and whether it contained the truth.

    vol_lo/vol_hi depend only on (alpha, beta) and on `volume_pred`, so they are rebuilt from the
    row rather than reused from the frame: two (alpha, beta) cells that share a lambda* must not
    share an interval, or the coverage column answers a question the header does not ask.
    Identical to safepp.volume_interval, verified case-by-case against it.
    """
    v, g = df["volume_pred"].values.astype(float), df["volume_gt"].values.astype(float)
    hi = v / max(1e-9, 1.0 - float(alpha)); lo = v * (1.0 - float(beta))
    lo2, hi2 = np.minimum(lo, hi), np.maximum(lo, hi)
    out = df.copy()
    out["vol_lo"], out["vol_hi"], out["vol_covered"] = lo2, hi2, (lo2 <= g) & (g <= hi2)
    return out


STRATA_TEST = {}


def stratum_ids(strata, which, ids):
    """Test cases on one side of the contrast cut (the cut is fixed by VALIDATION)."""
    if not len(man) or "wt_cnr" not in man.columns:
        return []
    lo = float(strata["cut"])
    return [p for p in ids if p in man.index and
            ((float(man.loc[p, "wt_cnr"]) <= lo) if which == "low"
             else (float(man.loc[p, "wt_cnr"]) > lo))]


if RUN_TEST:
    print("=" * 88)
    print("CERTIFIED VOLUME TEST -- do the promises survive untouched cases?")
    print("=" * 88)
    for arm in ARMS:
        df, c = TEST_DF[arm], CERT[arm]
        print(f"\n[{arm}] region {PRIMARY.upper()} | alpha={c['alpha']:.2f} (missed) "
              f"beta={c['beta']:.2f} (wrong) | lambda*={c['lam']:.2f} | n_cal={c['n_cal']}")
        # these are the per-case means; their standard errors are printed with the coverage table,
        # because at n_test=37 a mean 0.01 above the promise is inside the noise of the mean itself,
        # let alone of the case distribution -- read them together and do not quote one alone.
        print(f"   realised missed-tumour fraction : {df.missed_frac.mean():.3f} "
              f"(target <= {c['alpha']:.2f})  {'OK' if df.missed_frac.mean() <= c['alpha'] + 0.02 else 'VIOLATED'}")
        print(f"   realised wrong-prediction share : {df.wrong_frac.mean():.3f} "
              f"(target <= {c['beta']:.2f})  {'OK' if df.wrong_frac.mean() <= c['beta'] + 0.03 else 'VIOLATED'}")
        print(f"   volume interval contained truth : {df.vol_covered.mean():.0%} of cases")
        print(f"   median Dice WT/TC/ET            : {df.dice_wt.median():.3f} / "
              f"{df.dice_tc.median():.3f} / {df.dice_et.median():.3f} "
              f"(raw, pre-post-processing WT: {df.dice_wt_raw.median():.3f})")
        print(f"   components: {df.n_components_raw.mean():.1f} -> {df.n_components.mean():.1f} "
              f"| false: {df.false_components_raw.mean():.1f} -> {df.false_components.mean():.1f} "
              f"| missed lesions {df.missed_lesions.mean():.1f}")
        if "strata" in c:
            st = c["strata"]
            print(f"   stratified lambda*: low {st['lam_low']} (n_cal={st['n_low']}) vs high "
                  f"{st['lam_high']} (n_cal={st['n_high']}) | pooled {c['lam']}")
            STRATA_TEST[arm] = {}
            for _n in ("low", "high"):
                _ids = stratum_ids(st, _n, TEST_IDS)
                _feas = bool(st["feasible_low" if _n == "low" else "feasible_high"])
                _lam = float(st["lam_low" if _n == "low" else "lam_high"])
                STRATA_TEST[arm][_n] = dict(n=len(_ids), lam=_lam, feasible=_feas)
                if not _ids:
                    continue
                if not _feas:
                    print(f"   {_n}-contrast test subgroup (n={len(_ids)}): no certificate at "
                          f"this alpha/beta -- report the pooled number and say so")
                    continue
                d2 = eval_subset(arm, _lam, _ids)
                ok_m = d2.missed_frac.mean() <= c["alpha"] + 0.02
                ok_w = d2.wrong_frac.mean() <= c["beta"] + 0.03
                print(f"   {_n}-contrast test subgroup (n={len(_ids)}) at its OWN lambda "
                      f"{_lam:.2f}: missed {d2.missed_frac.mean():.3f} <= {c['alpha']:.2f} "
                      f"{'OK' if ok_m else 'VIOLATED'} | wrong {d2.wrong_frac.mean():.3f} <= "
                      f"{c['beta']:.2f} {'OK' if ok_w else 'VIOLATED'} | "
                      f"covered {d2.vol_covered.mean():.0%} | Dice WT {d2.dice_wt.median():.3f}")
                STRATA_TEST[arm][_n].update(missed=float(d2.missed_frac.mean()),
                                            wrong=float(d2.wrong_frac.mean()),
                                            covered=float(d2.vol_covered.mean()),
                                            dice_wt=float(d2.dice_wt.median()))

    # certified component filter: how aggressively may we delete fragments?
    print("\nCERTIFIED COMPONENT FILTER (collateral damage = true tumour a filter removes)")
    for arm in ARMS:
        RC = []
        for pid in VAL_IDS:
            z = np.load(os.path.join(PROB_DIRS[arm], f"{pid}.npz"))
            seg = np.load(bratsx.cache_file(CACHE_BASE, pid, "seg"))
            raw, post = apply_pipeline(z, CERT[arm]["lam"], POST)
            RC.append(safepp.removal_curve(post, seg, np.array(FILTER_GRID, float), region=PRIMARY.upper()))
        RC = np.stack(RC)
        for a in FILTER_ALPHAS:
            f = safepp.certified_filter_threshold(RC, alpha=a, thresholds=np.array(FILTER_GRID, float),
                                                  component_size_floor=min(FILTER_GRID[-2], 300.0))
            print(f"   [{arm}] alpha={a:.2f} -> delta={int(f['threshold'])} voxels "
                  f"(certified {int(f['certified_threshold'])}, bound {f['bound_cal']:.3f})")

    # Bland-Altman: does post-processing shift the volume error?
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        arm = ARMS[0]; df = TEST_DF[arm]
        fig, ax = plt.subplots(1, 3, figsize=(13, 4), dpi=120)
        raw_err = df.volume_err_raw.values     # already computed per case by evaluate()
        post_err = df.volume_err.values
        mean_v = (raw_err + post_err) / 2
        ax[0].scatter(mean_v, post_err - raw_err, s=18, c="#58a6ff")
        m_, sd_ = float(np.mean(post_err - raw_err)), float(np.std(post_err - raw_err))
        ax[0].axhline(m_, color="#3fb950"); ax[0].axhline(m_ + 1.96 * sd_, color="#d29922", ls="--")
        ax[0].axhline(m_ - 1.96 * sd_, color="#d29922", ls="--")
        ax[0].set_title(f"Bland-Altman: post-processing effect on {PRIMARY.upper()} volume")
        ax[0].set_xlabel("mean volume (mm3)"); ax[0].set_ylabel("post - raw (mm3)")
        ax[1].hist(df.missed_frac, bins=12, color="#f0883e")
        ax[1].axvline(CERT[arm]["alpha"], color="#3fb950", lw=1.5)
        ax[1].set_title(f"realised missed fraction (certified at {CERT[arm]['alpha']:.2f})")
        ax[2].hist(df.wrong_frac, bins=12, color="#58a6ff")
        ax[2].axvline(CERT[arm]["beta"], color="#3fb950", lw=1.5)
        ax[2].set_title(f"realised wrong-prediction share (certified at {CERT[arm]['beta']:.2f})")
        plt.tight_layout(); plt.savefig(os.path.join(WORK, "certified_summary.png"), dpi=120)
        print("\nfigure ->", os.path.join(WORK, "certified_summary.png"))
    except Exception as e:
        print("figure skipped:", e)

    # the full (alpha, beta) trade-off, and the certified filter, as CSV for N3
    rows = []
    for arm in ARMS:
        for a in ALPHAS:
            for b in BETAS:
                g = CERT[arm]["grid"].get(f"{float(a):.2f}|{float(b):.2f}")
                if not g or g["lam"] is None:
                    rows.append(dict(arm=arm, alpha=a, beta=b, lam=np.nan, reflected=np.nan,
                                     realised_missed=np.nan, realised_missed_se=np.nan,
                                     realised_wrong=np.nan, realised_wrong_se=np.nan,
                                     containment=np.nan)); continue
                # the interval is rebuilt FROM THIS CELL's alpha/beta: [v(1-b), v/(1-a)]
                df = promise_columns(eval_subset(arm, g["lam"], TEST_IDS), a, b)
                # a realised mean is a measurement, not a guarantee: its own noise is sd/sqrt(n),
                # and at n_test=37 that is the difference between "held" and "held by luck"
                _se_m = float(df.missed_frac.std(ddof=1) / np.sqrt(len(df)))
                _se_w = float(df.wrong_frac.std(ddof=1) / np.sqrt(len(df)))
                rows.append(dict(arm=arm, alpha=a, beta=b, lam=g["lam"],
                                 feasible=bool(g["feasible"]),
                                 # the cell the certificate is ACTUALLY at: every row of an arm is
                                 # the same mask, so a non-reflected row does not describe that mask
                                 reflected=bool(abs(float(g["lam"]) - float(CERT[arm]["lam"])) < 1e-12),
                                 realised_missed=float(df.missed_frac.mean()),
                                 realised_missed_se=_se_m,
                                 realised_wrong=float(df.wrong_frac.mean()),
                                 realised_wrong_se=_se_w,
                                 containment=float(df.vol_covered.mean())))
    ct = pd.DataFrame(rows)
    ct.to_csv(os.path.join(CSV_DIR, "coverage_table.csv"), index=False)
    print("\ncoverage trade-off on TEST.  Only `reflected=True` rows describe the DELIVERED mask "
          "(they carry the arm's own lambda*); every other row is the same mask scored against a "
          "promise it was not certified at -- exploratory, never a headline.")
    print(ct.round(3).to_string(index=False))
    for arm in ARMS:
        _r = ct[(ct.arm == arm) & ct.reflected.astype(bool)]
        if len(_r):
            _r = _r.iloc[0]
            print(f"   [{arm}] delivered cell alpha={_r.alpha:.2f} beta={_r.beta:.2f}: "
                  f"realised missed {_r.realised_missed:.3f} +/- {_r.realised_missed_se:.3f} (se), "
                  f"wrong {_r.realised_wrong:.3f} +/- {_r.realised_wrong_se:.3f} (se) "
                  f"over {len(TEST_IDS)} test cases")

    # certified filter + the per-arm summary N3 renders
    summary = {}
    for arm in ARMS:
        RC = []
        for pid in VAL_IDS:
            z = np.load(os.path.join(PROB_DIRS[arm], f"{pid}.npz"))
            seg = np.load(bratsx.cache_file(CACHE_BASE, pid, "seg"))
            raw, post = apply_pipeline(z, CERT[arm]["lam"], POST)
            RC.append(safepp.removal_curve(post, seg, np.array(FILTER_GRID, float),
                                           region=PRIMARY.upper()))
        RC = np.stack(RC)
        fa = float(FILTER_ALPHAS[0]) if FILTER_ALPHAS else 0.05
        f = safepp.certified_filter_threshold(RC, alpha=fa, thresholds=np.array(FILTER_GRID, float),
                                              component_size_floor=min(FILTER_GRID[-2], 300.0))
        summary[arm] = dict(alpha=float(CERT[arm]["alpha"]), beta=float(CERT[arm]["beta"]),
                            lam=float(CERT[arm]["lam"]), cell_shared=bool(CERT[arm]["cell_shared"]),
                            cell_note=CERT[arm]["cell_note"],
                            filter_alpha=fa, filter_delta=int(f["threshold"]),
                            filter_certified=int(f["certified_threshold"]),
                            filter_bound=float(f["bound_cal"]),
                            # does a component floor ABOVE 1 voxel exist at all?  At
                            # alpha=0.01 with n calibration cases the finite-sample term alone
                            # is 1/(n+1), so below n=99 the answer is always "no" and the
                            # delivered mask is the plain thresholded mask.
                            filter_feasible=bool(f["feasible"]),
                            n_test=len(TEST_IDS),
                            # the certificate is about WT.  These are the same mask's TC/ET scores
                            # and are NOT covered by the WT guarantee -- label them that way
                            # wherever they are shown.
                            med_dice_wt=float(TEST_DF[arm].dice_wt.median()),
                            med_dice_tc=float(TEST_DF[arm].dice_tc.median()),
                            med_dice_et=float(TEST_DF[arm].dice_et.median()),
                            missed=float(TEST_DF[arm].missed_frac.mean()),
                            missed_se=float(TEST_DF[arm].missed_frac.std(ddof=1)
                                            / np.sqrt(max(1, len(TEST_DF[arm])))),
                            wrong=float(TEST_DF[arm].wrong_frac.mean()),
                            wrong_se=float(TEST_DF[arm].wrong_frac.std(ddof=1)
                                           / np.sqrt(max(1, len(TEST_DF[arm])))),
                            containment=float(TEST_DF[arm].vol_covered.mean()),
                            strata=STRATA_TEST.get(arm))
    json.dump(summary, open(os.path.join(WORK, "summary.json"), "w"), indent=1)
    print("\nsummary.json written (N3 reads it for the report headers)")

    print("""
HOW TO READ THIS (put these three lines on the slide)
  * `realised missed` below `alpha` = the certificate held on cases the calibration never saw;
    the gap between them is the price of the finite-sample correction, not a failure.  A mean a
    little ABOVE alpha at n_test=37 is not proof of a violation either: check it against its
    standard error, which the coverage table prints next to it.
  * `volume interval contained truth` is the clinical number: the share of scans whose true
    volume fell inside the interval the tool displays.
  * `lam_low < lam_high` means the tool automatically becomes more conservative on the
    low-contrast scans -- that is the point of stratifying the calibration.
""")


In [ ]:
# %% [CELL 10] PACK THE RUN FOR CHECKING (automatic) --------------------------
# This runs by itself at the end of N2 so that nothing has to be remembered or typed: the zip
# that a reviewer needs is simply THERE in the output pane.  (collect_report.py does the same
# thing by hand if you ever want to re-pack a finished session.)
import glob as _glob, zipfile as _zip

_PATTERNS = ["certificates.json", "summary.json", "frozen_postproc.json", "post_grid.csv",
             "strata.csv", "nonfinite_cases.json", "certified_summary.png", "n2_log.txt",
             "test_csvs/*.csv", "test_csvs/*.json"]
_files = []
for _pat in _PATTERNS:
    _files += sorted(_glob.glob(os.path.join(WORK, _pat)))
_zip_path = os.path.join(WORK, "n2_report.zip")
if _files:
    with _zip.ZipFile(_zip_path, "w", _zip.ZIP_DEFLATED) as _z:
        for _f in _files:
            _z.write(_f, os.path.relpath(_f, WORK))
    print(f"\n[pack] {_zip_path}: {len(_files)} file(s), {os.path.getsize(_zip_path):,} bytes")
    for _f in sorted(os.path.relpath(f, WORK) for f in _files):
        print(f"[pack]    {_f}")
    _missing = [q for q in ("certificates.json", "summary.json", "frozen_postproc.json")
                if not _glob.glob(os.path.join(WORK, q))]
    if _missing:
        print(f"[pack] !! {_missing} missing -- this run cannot be checked without them")
    else:
        print("[pack] download n2_report.zip from the output pane -- that single file is "
              "everything a reviewer needs")
else:
    print(f"[pack] nothing to pack in {WORK}: did the evaluation cells run?")


In [ ]:
# %% [CELL 11] NEXT ---------------------------------------------------------
print("""
NEXT
  1. Output of this notebook = test_csvs/, certificates.json, frozen_postproc.json and
     (if you ran it) the cached probs/ and certified_summary.png -> Create a dataset.
  2. N3_report reads probs/ + certificates.json + frozen_postproc.json + the cache and
     builds the interactive per-case report.  CPU only; re-runnable as often as you like.
  3. Nothing here is re-tuned after seeing test.  If you change POST or the alpha/beta
     targets, treat the test numbers as exploratory and say so.
""")
